**PYTHON**

**DECORATORS**

*The Complete Handbook - From Beginner to Professional*

A Comprehensive Teaching and Reference Guide to Understanding, Building, and Applying Python Decorators in Data Science, Machine Learning, AI Engineering, and Software Development

**Kajola Gbenga**

*Data Scientist & AI Engineer*

**Table of Contents**

1. Introduction to Decorators

2. First Principles Behind Decorators

3. Creating Your First Decorator

4. The @decorator Syntax

5. Decorators for Functions with Arguments

6. Returning Values from Decorated Functions

7. Preserving Function Metadata with functools.wraps

8. Practical Decorator Patterns

9. Decorators with Arguments

10. Stacking Multiple Decorators

11. Class-Based Decorators

12. Decorators for Classes

13. Built-in Python Decorators

14. property, classmethod, and staticmethod as Decorator Concepts

15. Decorators and Closures

16. Decorators and Scope

17. Advanced functools Decorators

18. Memoization and Caching

19. Decorators for Data Science

20. Decorators for Machine Learning and AI Engineering

21. Decorators in Web Development

22. Decorators and Asynchronous Python

23. Writing Production-Quality Decorators

24. Common Decorator Mistakes

25. Debugging Decorators

26. Testing Decorators

27. Decorator Performance

28. Professional Real-World Projects

29. Decorators Interview Preparation

30. Practical Exercises

31. Decorators Cheat Sheet and Quick Reference


# 1. Introduction to Decorators

## 1.1 What a Decorator Is

A **decorator** is a function that takes another function (or class) as input and returns a new function (or class) with extended or modified behavior, without permanently changing the original's source code. In Python, decorators are applied using the `@` symbol placed directly above a function or class definition.

In [ ]:
@my_decorator
def greet():
    print("Hello")

## 1.2 Why Decorators Exist

Real programs frequently need to add the *same* extra behavior to many different functions: logging every call, timing execution, checking permissions, validating input, or retrying on failure. Without decorators, a developer would have to manually copy that extra logic into every single function that needs it, or wrap every call site by hand. Decorators solve this by letting that shared behavior be written **once** and attached to any function with a single line.

## 1.3 The Problem Decorators Solve

In [ ]:
def process_order(order):
    print("Starting timer")
    import time
    start = time.time()
    result = order * 2  # actual work
    print(f"Took {time.time() - start:.4f}s")
    return result

def process_refund(refund):
    print("Starting timer")
    import time
    start = time.time()
    result = refund * -1  # actual work
    print(f"Took {time.time() - start:.4f}s")
    return result

Both functions repeat identical timing logic around different core work. This is a direct violation of the "don't repeat yourself" (DRY) principle. A decorator extracts the repeated part into a single, reusable piece:

In [ ]:
import time
from functools import wraps

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"Took {time.time() - start:.4f}s")
        return result
    return wrapper

@timed
def process_order(order):
    return order * 2

@timed
def process_refund(refund):
    return refund * -1

## 1.4 Decorators in Simple Terms

A decorator is best understood as a **wrapper**: it takes your function, places new behavior *before* and/or *after* it, and hands back something that looks and behaves like your original function to the caller, but does more.

## 1.5 A Real-World Analogy

Think of a decorator like a gift-wrapping service. You hand over a plain gift (the original function). The service wraps it in paper and a ribbon (the decorator's added behavior) without changing what's inside the box (the function's core logic). The recipient (the caller) still receives a gift they can open the same way - but now it also looks a certain way, or comes with a card attached, depending on the wrapping applied.

## 1.6 Relationship Between Decorators and Functions

A decorator is itself an ordinary Python function (or occasionally a class, Part 11). What makes it a "decorator" is not special syntax inside it, but the *role* it plays: it accepts a function as an argument and returns a (usually different) function as its result.

## 1.7 Modifying Behavior Without Changing Source Code

The original function's source code - its own body, its own logic - is never edited. The decorator instead builds a new function *around* it. This separation is what makes decorators so powerful for cross-cutting concerns (logging, timing, security) that apply to many unrelated functions without polluting each one's core logic.

## 1.8 Why Decorators Matter in Professional Python

| Reason | Explanation |
|---|---|
| Eliminates repetition | Shared behavior is written once, applied everywhere it's needed. |
| Separation of concerns | Business logic stays clean; cross-cutting behavior (logging, timing, auth) lives separately. |
| Framework integration | Flask, FastAPI, Django, pytest, and most professional frameworks are built around decorators as their primary extension mechanism. |
| Readability | `@login_required` above a function communicates intent far more clearly than an equivalent `if` check buried inside it. |

## 1.9 Common Places Decorators Are Used

| Context | Example |
|---|---|
| Web frameworks | `@app.route("/users")` in Flask/FastAPI |
| Testing | `@pytest.fixture`, `@pytest.mark.parametrize` |
| Object-oriented code | `@property`, `@staticmethod`, `@classmethod` |
| Performance | `@functools.lru_cache` |
| Data pipelines | Custom `@log_step`, `@validate_input` decorators |
| Machine learning | Custom `@track_experiment`, `@time_inference` decorators |


# 2. First Principles Behind Decorators

## 2.1 Functions as Objects

In Python, a function is not special syntax detached from the rest of the language - it is an **object**, just like an integer, a string, or a list. It can be assigned to a variable, stored in a data structure, passed around, and inspected at runtime.

In [ ]:
def greet():
    return "Hello"

print(type(greet))   # <class 'function'>

## 2.2 Assigning Functions to Variables

In [ ]:
def greet():
    return "Hello"

say_hi = greet   # no parentheses - assigns the function object itself
print(say_hi())  # "Hello"

`say_hi` and `greet` now both refer to the exact same function object in memory. Calling either one runs identical code.

## 2.3 The Critical Distinction: function vs function()

| Expression | What It Refers To |
|---|---|
| `greet` | The function object itself - a reference, not a call. |
| `greet()` | The *result* of calling the function - executes the body and produces a value. |

In [ ]:
print(greet)     # <function greet at 0x...>  - the object
print(greet())   # "Hello"                    - the object, called

This distinction is the single most important prerequisite for understanding decorators: a decorator receives the function *object* (`greet`, no parentheses) - never a call result - and it typically returns a *different* function object.

## 2.4 Passing Functions as Arguments

In [ ]:
def shout(text):
    return text.upper() + "!"

def announce(formatter, message):
    print(formatter(message))

announce(shout, "hello")   # HELLO!

`shout` is passed into `announce` as a plain value - the same way a number or string would be - and `announce` calls it internally using `formatter(message)`.

## 2.5 Returning Functions from Functions

In [ ]:
def make_multiplier(factor):
    def multiplier(number):
        return number * factor
    return multiplier   # returns a function object, not a call result

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15

`make_multiplier` builds and returns a brand new function each time it is called, with `factor` "baked in" - this is the foundation of closures, covered fully in Part 15.

## 2.6 Nested Functions

A function defined inside another function is called a **nested function** (or inner function). It exists only within the scope of the outer function unless it is explicitly returned or otherwise exposed.

In [ ]:
def outer():
    def inner():
        return "I'm nested"
    return inner()

print(outer())   # "I'm nested"

## 2.7 Function References vs Function Calls

A **function reference** is simply a name bound to a function object (`greet`). A **function call** executes that object's code (`greet()`). Confusing the two is one of the most common beginner mistakes in decorator code - accidentally writing `func()` where `func` was intended, or vice versa.

## 2.8 Higher-Order Functions

A **higher-order function** is any function that does at least one of the following: accepts another function as an argument, or returns a function as its result. `make_multiplier` above is a higher-order function because it returns one. `announce` above is a higher-order function because it accepts one.

## 2.9 First-Class Functions

Python is said to treat functions as **first-class citizens**, meaning functions can be used anywhere any other value can be used: assigned to variables, stored in lists or dictionaries, passed as arguments, and returned from other functions.

In [ ]:
operations = {
    "double": lambda x: x * 2,
    "square": lambda x: x ** 2,
}

print(operations["square"](4))   # 16

## 2.10 Why These Concepts Make Decorators Possible

A decorator is, at its core, just a higher-order function: it **accepts** a function (first-class functions make this possible), builds a **nested function** around it (nested functions + closures), and **returns** that new function (functions returned from functions). Every single decorator you will ever write or read rests entirely on these four ideas working together.


# 3. Creating Your First Decorator

## 3.1 Starting Point: A Plain Function

In [ ]:
def greet():
    print("Hello")

greet()

## 3.2 Building a Decorator Step by Step

**Step 1 - write a function that accepts another function:**

In [ ]:
def shout_decorator(func):
    pass

**Step 2 - define a wrapper function inside it:**

In [ ]:
def shout_decorator(func):
    def wrapper():
        print("Something is about to happen...")
        func()
        print("Something happened.")
    return wrapper

**Step 3 - apply it manually:**

In [ ]:
def greet():
    print("Hello")

greet = shout_decorator(greet)
greet()

```
Something is about to happen...
Hello
Something happened.
```

## 3.3 How the Wrapper Works

| Piece | Role |
|---|---|
| `func` | The original function, passed into the decorator as a parameter. |
| `wrapper` | A new, nested function that calls `func()` somewhere inside it, surrounded by extra behavior. |
| `return wrapper` | The decorator hands back the wrapper - not the original function - to replace it. |

## 3.4 How the Original Function Is Passed Into the Decorator

When you write `shout_decorator(greet)`, the function object `greet` (not a call - see Part 2.3) is passed as the argument `func`. Inside `shout_decorator`, `func` is simply another name referring to that same original function object.

## 3.5 How the Decorator Returns the Wrapper

`shout_decorator` does not call `wrapper()` itself - it **returns the wrapper function object**, unexecuted. The caller decides when to actually run it, exactly like calling any other function.

## 3.6 What Python Actually Does With @decorator Syntax

In [ ]:
@shout_decorator
def greet():
    print("Hello")

is **exactly equivalent** to:

In [ ]:
def greet():
    print("Hello")

greet = shout_decorator(greet)

| Step | What Happens |
|---|---|
| 1 | Python defines `greet` normally, as an ordinary function object. |
| 2 | Python calls `shout_decorator(greet)`, passing in that function object. |
| 3 | `shout_decorator` returns `wrapper`. |
| 4 | Python rebinds the name `greet` to point at `wrapper` instead of the original function. |

## 3.7 Calling the Decorated Function

In [ ]:
greet()

Because `greet` now refers to `wrapper`, calling `greet()` actually runs `wrapper()`, which in turn calls the *original* function (now only reachable through the closure variable `func` inside `wrapper`) at the appropriate point.

## 3.8 The Complete, Final Example

In [ ]:
def shout_decorator(func):
    def wrapper():
        print("Something is about to happen...")
        func()
        print("Something happened.")
    return wrapper

@shout_decorator
def greet():
    print("Hello")

greet()

```
Something is about to happen...
Hello
Something happened.
```


# 4. The @decorator Syntax

## 4.1 Basic @decorator Syntax

In [ ]:
@my_decorator
def my_function():
    ...

## 4.2 The Equivalent Assignment Syntax

In [ ]:
def my_function():
    ...
my_function = my_decorator(my_function)

Both forms are identical in effect; `@decorator` is purely syntactic sugar that Python provides to make this extremely common pattern more readable and less repetitive.

## 4.3 Decorating Multiple Functions

In [ ]:
@timed
def load_data():
    ...

@timed
def clean_data():
    ...

@timed
def train_model():
    ...

The same decorator can be applied to as many independent functions as needed, each producing its own separately wrapped version.

## 4.4 Applying Different Decorators to Different Functions

In [ ]:
@timed
def load_data():
    ...

@requires_admin
def delete_all_records():
    ...

@cached
def compute_expensive_result():
    ...

## 4.5 Multiple Decorators on One Function

In [ ]:
@decorator_one
@decorator_two
def my_function():
    ...

## 4.6 Decorator Order: Bottom-Up Application

Decorators are **applied** from the bottom upward, even though they are read top to bottom. The equivalent assignment form makes this explicit:

In [ ]:
def my_function():
    ...
my_function = decorator_one(decorator_two(my_function))

`decorator_two` wraps the original function *first*; `decorator_one` then wraps the *already-wrapped* result.

## 4.7 Execution Order vs Application Order

| Concept | Order |
|---|---|
| Application (decoration) order | Bottom-up: the decorator closest to the function runs first, wrapping it. |
| Execution (call) order | Top-down: when the final decorated function is called, the outermost wrapper's code runs first. |

In [ ]:
def decorator_one(func):
    def wrapper(*args, **kwargs):
        print("One: before")
        result = func(*args, **kwargs)
        print("One: after")
        return result
    return wrapper

def decorator_two(func):
    def wrapper(*args, **kwargs):
        print("Two: before")
        result = func(*args, **kwargs)
        print("Two: after")
        return result
    return wrapper

@decorator_one
@decorator_two
def my_function():
    print("Function body")

my_function()

```
One: before
Two: before
Function body
Two: after
One: after
```

## 4.8 Visualizing the Nesting

```
decorator_one(
    decorator_two(
        my_function
    )
)
```

Calling the fully decorated `my_function()` is like peeling an onion from the outside in: `decorator_one`'s wrapper runs first (its "before" logic), then hands control to `decorator_two`'s wrapper (its "before" logic), which finally calls the real function - and then control unwinds back outward through each wrapper's "after" logic, in reverse.


# 5. Decorators for Functions with Arguments

## 5.1 The Problem: The Simple Wrapper Only Works for No-Argument Functions

In [ ]:
def shout_decorator(func):
    def wrapper():
        func()
    return wrapper

@shout_decorator
def add(a, b):
    print(a + b)

add(2, 3)

```
TypeError: wrapper() takes 0 positional arguments but 2 were given
```

The wrapper from Part 3 has no parameters, so it cannot accept `a` and `b` at all - and even if it did, it would need to forward them on to `func`.

## 5.2 The Fix: `*args` and `**kwargs`

In [ ]:
def shout_decorator(func):
    def wrapper(*args, **kwargs):
        print("Before")
        result = func(*args, **kwargs)
        print("After")
        return result
    return wrapper

@shout_decorator
def add(a, b):
    print(a + b)

add(2, 3)

```
Before
5
After
```

## 5.3 Why Professional Decorators Almost Always Use `*args`, `**kwargs`

| Argument Style Supported | Example Call | Handled By |
|---|---|---|
| No arguments | `func()` | `*args` and `**kwargs` both empty |
| Positional arguments | `func(1, 2)` | Captured in `args` |
| Keyword arguments | `func(a=1, b=2)` | Captured in `kwargs` |
| Mixed | `func(1, b=2)` | Split correctly between `args` and `kwargs` |
| Arbitrary/unknown arguments | Any signature at all | Works without modification |

Using `*args, **kwargs` in the wrapper means the *same* decorator works correctly on **any** function, regardless of how many parameters it has or how they are called - which is essential, since a decorator is usually written once and applied to many different functions with different signatures.

## 5.4 Practical Examples Across Argument Styles

In [ ]:
@shout_decorator
def no_args():
    print("no arguments")

@shout_decorator
def positional(a, b, c):
    print(a, b, c)

@shout_decorator
def keyword_only(*, name, age):
    print(name, age)

@shout_decorator
def mixed(a, b, *, c):
    print(a, b, c)

no_args()
positional(1, 2, 3)
keyword_only(name="Ada", age=30)
mixed(1, 2, c=3)

Each call works correctly without the decorator needing any knowledge of the specific function's signature - `*args, **kwargs` simply passes through whatever was received.


# 6. Returning Values from Decorated Functions

## 6.1 Why Wrappers Must Properly Return Values

If a wrapper calls `func(*args, **kwargs)` but never returns that result, the decorated function silently becomes one that always returns `None` - a bug that can be very difficult to trace, since no error is raised.

## 6.2 The Correct Pattern

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result
    return wrapper

## 6.3 The Common Mistake

In [ ]:
def broken_decorator(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)   # result is computed but never returned!
    return wrapper

@broken_decorator
def add(a, b):
    return a + b

print(add(2, 3))   # None  <- the real answer, 5, is silently lost

| Stage | Detail |
|---|---|
| Symptom | The decorated function always returns `None`, regardless of what the original function actually computed. |
| Cause | The wrapper calls `func(*args, **kwargs)` but discards the result instead of returning it. |
| Fix | Capture the result in a variable and `return` it explicitly (or `return func(*args, **kwargs)` directly). |

## 6.4 Demonstrating With Different Return Types

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result
    return wrapper

@my_decorator
def get_number():
    return 42

@my_decorator
def get_text():
    return "hello"

@my_decorator
def get_list():
    return [1, 2, 3]

@my_decorator
def get_dict():
    return {"status": "ok"}

print(get_number())   # 42
print(get_text())     # hello
print(get_list())     # [1, 2, 3]
print(get_dict())     # {'status': 'ok'}

Because the wrapper always captures and returns `func`'s result unchanged, the decorator is completely transparent with respect to return values - the caller gets back exactly what the original function would have produced, regardless of its type.

## 6.5 A Data-Processing Example

In [ ]:
def logged(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result!r}")
        return result
    return wrapper

@logged
def clean_row(row):
    return {k: v.strip() for k, v in row.items()}

cleaned = clean_row({"name": "  Ada  ", "city": " Lagos "})
print(cleaned)

```
Calling clean_row
clean_row returned {'name': 'Ada', 'city': 'Lagos'}
{'name': 'Ada', 'city': 'Lagos'}
```

Here `cleaned` correctly holds the real dictionary returned by `clean_row`, confirming the wrapper passed the return value through intact.


# 7. Preserving Function Metadata with functools.wraps

## 7.1 The Problem: Decorators Silently Replace Function Identity

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def calculate_total(prices):
    """Return the sum of a list of prices."""
    return sum(prices)

print(calculate_total.__name__)   # 'wrapper'   <- wrong!
print(calculate_total.__doc__)    # None         <- wrong!

Because `calculate_total` now actually refers to `wrapper`, every piece of introspection metadata - its name, its docstring, its module - reflects the generic wrapper, not the original function.

## 7.2 Why Metadata Matters

| Attribute | What It's Used For |
|---|---|
| `__name__` | Debugging output, logging, error messages, and introspection tools all rely on a function's real name. |
| `__doc__` | Documentation generators (Sphinx), `help()`, and IDEs display this to describe what a function does. |
| `__module__` | Identifies which module a function was defined in - used by debuggers and profilers. |
| `__annotations__` | Type hints, used by static type checkers like `mypy` and by some frameworks at runtime. |
| `__wrapped__` | A reference back to the original, undecorated function - lets introspection tools "see through" the decorator. |

## 7.3 Problems Caused By Losing Metadata

Losing metadata is not just cosmetic - it actively breaks tooling: automatic API documentation generators produce entries labeled `wrapper` for every decorated endpoint; debuggers show unhelpful stack traces; and other decorators or frameworks that inspect a function's signature (Part 25) can behave incorrectly.

## 7.4 The Solution: functools.wraps

In [ ]:
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def calculate_total(prices):
    """Return the sum of a list of prices."""
    return sum(prices)

print(calculate_total.__name__)   # 'calculate_total'  correct
print(calculate_total.__doc__)    # 'Return the sum of a list of prices.'  correct

## 7.5 How @wraps Works

`functools.wraps(func)` is itself a decorator *factory* (Part 9) that, when applied to `wrapper`, copies over `func`'s `__name__`, `__doc__`, `__module__`, and `__annotations__` onto `wrapper`, and additionally sets `wrapper.__wrapped__ = func`, preserving a direct reference to the original.

In [ ]:
print(calculate_total.__wrapped__)   # <function calculate_total at 0x...> - the original, unwrapped function

## 7.6 Comparing With and Without @wraps

| Without @wraps | With @wraps |
|---|---|
| `func.__name__` becomes `"wrapper"` | `func.__name__` correctly stays the original name |
| `func.__doc__` is lost (`None`) | `func.__doc__` is preserved |
| No way to reach the original function | `func.__wrapped__` gives direct access to it |
| Debugging and documentation tools are misled | Debugging and documentation tools work correctly |

## 7.7 Professional Guidance

**Every** decorator you write professionally should apply `@wraps(func)` to its inner wrapper function, with no exceptions, unless you have an unusual, deliberate reason not to preserve the original identity. It costs one line and prevents an entire category of subtle, hard-to-diagnose bugs and tooling failures.


# 8. Practical Decorator Patterns

Each pattern below follows the same structure: the problem it solves, the code, a line-by-line explanation, a real-world use case, and common mistakes to avoid. All wrappers use `@wraps` (Part 7) and `*args, **kwargs` (Part 5) as a baseline professional standard.

## 8.1 Logging

**Problem:** You want visibility into when functions are called and with what arguments, without adding print statements inside every function.

In [ ]:
import logging
from functools import wraps

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        logger.info(f"{func.__name__} returned {result!r}")
        return result
    return wrapper

@log_calls
def add(a, b):
    return a + b

add(2, 3)

| Line | Explanation |
|---|---|
| `logger.info(...)` before the call | Records the function name and every argument passed, before execution. |
| `result = func(*args, **kwargs)` | Executes the real function, capturing its result. |
| `logger.info(...)` after the call | Records the outcome, useful for tracing program flow after the fact. |

**Real-world use case:** tracing a data pipeline's execution across many stages in production logs. **Common mistake:** logging sensitive arguments (passwords, tokens) without redaction.

## 8.2 Timing Functions

**Problem:** You need to measure how long specific functions take to run, consistently, across a codebase.

In [ ]:
import time
from functools import wraps

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timed
def slow_computation():
    time.sleep(0.5)
    return "done"

**Real-world use case:** identifying slow steps in an ML training pipeline. **Common mistake:** using `time.time()` instead of `time.perf_counter()`, which is less precise for short-duration measurements and can be affected by system clock adjustments.

## 8.3 Authentication

**Problem:** Certain functions (or API endpoints) should only run for an authenticated user.

In [ ]:
from functools import wraps

class AuthError(Exception):
    pass

def requires_auth(func):
    @wraps(func)
    def wrapper(user, *args, **kwargs):
        if not user.get("is_authenticated"):
            raise AuthError("Authentication required")
        return func(user, *args, **kwargs)
    return wrapper

@requires_auth
def view_profile(user):
    return f"Profile for {user['name']}"

view_profile({"name": "Ada", "is_authenticated": True})

**Real-world use case:** protecting API routes in Flask/FastAPI (Part 21). **Common mistake:** checking authentication *after* performing side-effecting work, rather than before.

## 8.4 Authorization

**Problem:** An authenticated user may still lack permission for a specific action.

In [ ]:
def requires_role(role):
    def decorator(func):
        @wraps(func)
        def wrapper(user, *args, **kwargs):
            if role not in user.get("roles", []):
                raise PermissionError(f"Requires role: {role}")
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

@requires_role("admin")
def delete_user(user, target_id):
    print(f"Deleted user {target_id}")

delete_user({"name": "Ada", "roles": ["admin"]}, target_id=42)

This uses the decorator-with-arguments pattern, explained fully in Part 9.

## 8.5 Validation (Input)

**Problem:** Ensure arguments meet expected conditions before a function runs.

In [ ]:
def validate_positive(func):
    @wraps(func)
    def wrapper(value, *args, **kwargs):
        if value <= 0:
            raise ValueError(f"{func.__name__} requires a positive value, got {value}")
        return func(value, *args, **kwargs)
    return wrapper

@validate_positive
def take_square_root(value):
    return value ** 0.5

**Common mistake:** validating only the first argument when multiple arguments need checking - be explicit about which ones the decorator actually covers.

## 8.6 Caching

**Problem:** Avoid recomputing the same expensive result multiple times.

In [ ]:
def simple_cache(func):
    cache = {}
    @wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper

@simple_cache
def slow_square(n):
    import time
    time.sleep(1)
    return n * n

slow_square(4)   # takes 1 second
slow_square(4)   # instant - cached

Covered in full depth, including `functools.lru_cache`, in Part 18.

## 8.7 Debugging

**Problem:** Trace exactly what arguments a function receives and what it returns, temporarily, without editing its body.

In [ ]:
def debug(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"DEBUG: {func.__name__}(args={args}, kwargs={kwargs})")
        result = func(*args, **kwargs)
        print(f"DEBUG: {func.__name__} -> {result!r}")
        return result
    return wrapper

## 8.8 Retry Mechanisms

**Problem:** Automatically retry a function that may fail transiently (e.g. a flaky network call).

In [ ]:
import time

def retry(max_attempts=3, delay=1):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"Attempt {attempt} failed: {e}. Retrying...")
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(max_attempts=3, delay=2)
def unstable_api_call():
    ...

**Common mistake:** retrying indefinitely with no maximum, or retrying errors that will never succeed on retry (like a validation error).

## 8.9 Performance Monitoring

**Problem:** Continuously track function performance across a running application, not just during local debugging.

In [ ]:
def monitor(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            send_metric(f"{func.__name__}.duration", elapsed)
    return wrapper

Using `finally` ensures the metric is recorded even if `func` raises an exception - important for production monitoring (Part 27).

## 8.10 Access Control

Closely related to authorization (8.4), but often applied at a coarser level - restricting an entire feature based on environment, subscription tier, or feature flag.

In [ ]:
def feature_flag(flag_name):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if not is_feature_enabled(flag_name):
                raise RuntimeError(f"Feature '{flag_name}' is disabled")
            return func(*args, **kwargs)
        return wrapper
    return decorator

## 8.11 Output Validation

**Problem:** Guarantee a function's return value meets an expected contract before it reaches the caller.

In [ ]:
def validate_output_type(expected_type):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            if not isinstance(result, expected_type):
                raise TypeError(f"{func.__name__} must return {expected_type}, got {type(result)}")
            return result
        return wrapper
    return decorator

@validate_output_type(dict)
def get_config():
    return {"debug": True}

## 8.12 Auditing

**Problem:** Record a permanent record of who performed a sensitive action and when.

In [ ]:
def audit(action_name):
    def decorator(func):
        @wraps(func)
        def wrapper(user, *args, **kwargs):
            result = func(user, *args, **kwargs)
            write_audit_log(user=user["name"], action=action_name, timestamp=time.time())
            return result
        return wrapper
    return decorator

## 8.13 Rate Limiting

**Problem:** Prevent a function (often an API endpoint) from being called more than a set number of times per interval.

In [ ]:
import time

def rate_limit(max_calls, period_seconds):
    def decorator(func):
        calls = []
        @wraps(func)
        def wrapper(*args, **kwargs):
            now = time.time()
            calls[:] = [t for t in calls if now - t < period_seconds]
            if len(calls) >= max_calls:
                raise RuntimeError("Rate limit exceeded")
            calls.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(max_calls=5, period_seconds=60)
def call_external_api():
    ...

## 8.14 Deprecation Warnings

**Problem:** Warn callers that a function will be removed in a future version, without breaking existing code immediately.

In [ ]:
import warnings

def deprecated(reason=""):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            warnings.warn(
                f"{func.__name__} is deprecated. {reason}",
                category=DeprecationWarning,
                stacklevel=2,
            )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@deprecated("Use new_function() instead.")
def old_function():
    ...

## 8.15 Monitoring Function Calls

**Problem:** Count how many times a function is called, useful for usage analytics or detecting unexpectedly hot code paths.

In [ ]:
def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def process_item(item):
    ...

process_item(1)
process_item(2)
print(process_item.calls)   # 2

This pattern attaches a custom attribute (`calls`) directly onto the wrapper function object itself - functions, being objects (Part 2.1), can freely hold extra attributes like this.


# 9. Decorators with Arguments

## 9.1 Why Some Decorators Need Their Own Arguments

A plain decorator like `@timed` takes no configuration - it always behaves the same way. But sometimes a decorator needs to be *configured* per use: how many retries, what role is required, what cache size. This requires an extra layer.

## 9.2 The Three-Layer Structure

In [ ]:
def decorator(argument):
    def actual_decorator(func):
        def wrapper(*args, **kwargs):
            # use "argument" and call func(*args, **kwargs)
            ...
        return wrapper
    return actual_decorator

| Layer | Role |
|---|---|
| `decorator(argument)` | The outermost function - accepts the decorator's *own* configuration arguments, and returns the real decorator. |
| `actual_decorator(func)` | The real decorator - accepts the function being decorated, and returns the wrapper. |
| `wrapper(*args, **kwargs)` | The innermost function - handles the actual call, with access to both `argument` (via closure) and `func` (via closure). |

## 9.3 Why Three Levels Are Required

In [ ]:
@repeat(3)
def greet():
    print("Hello")

This is equivalent to:

In [ ]:
greet = repeat(3)(greet)

`repeat(3)` must first execute and produce something - and that something is what actually receives `greet`. If `repeat` only had two layers (like a normal decorator), there would be no way to pass in `3` *before* `greet` becomes available.

## 9.4 A Complete Worked Example: @repeat(n)

In [ ]:
from functools import wraps

def repeat(times):
    def actual_decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return actual_decorator

@repeat(3)
def greet():
    print("Hello")

greet()

```
Hello
Hello
Hello
```

## 9.5 Configurable Logging

In [ ]:
def log_with_level(level="INFO"):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"[{level}] Calling {func.__name__}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@log_with_level("DEBUG")
def compute():
    ...

## 9.6 Configurable Retry Counts

In [ ]:
@retry(max_attempts=5, delay=2)
def flaky_call():
    ...

(Full implementation in Part 8.8.)

## 9.7 Threshold Decorators

In [ ]:
def alert_if_slow(threshold_seconds):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = func(*args, **kwargs)
            elapsed = time.perf_counter() - start
            if elapsed > threshold_seconds:
                print(f"WARNING: {func.__name__} took {elapsed:.2f}s (threshold {threshold_seconds}s)")
            return result
        return wrapper
    return decorator

@alert_if_slow(threshold_seconds=1.0)
def query_database():
    ...

## 9.8 Permission Decorators (Recap)

See Part 8.4 (`requires_role`) - a direct application of this same three-layer pattern.

## 9.9 Validation Decorators With Configuration

In [ ]:
def validate_range(minimum, maximum):
    def decorator(func):
        @wraps(func)
        def wrapper(value, *args, **kwargs):
            if not (minimum <= value <= maximum):
                raise ValueError(f"{value} out of range [{minimum}, {maximum}]")
            return func(value, *args, **kwargs)
        return wrapper
    return decorator

@validate_range(0, 100)
def set_percentage(value):
    print(f"Set to {value}%")

## 9.10 A Subtle Trap: Decorators With Optional Arguments

Writing a decorator that works both as `@my_decorator` and `@my_decorator(option=True)` requires detecting whether it was called with a function directly or with configuration first:

In [ ]:
def flexible_decorator(func=None, *, option=False):
    def decorator(f):
        @wraps(f)
        def wrapper(*args, **kwargs):
            if option:
                print("Option enabled")
            return f(*args, **kwargs)
        return wrapper
    if func is not None:
        return decorator(func)   # used as @flexible_decorator (no parentheses)
    return decorator             # used as @flexible_decorator(option=True)

This dual-mode pattern appears in some professional libraries but adds real complexity - use it only when the ergonomic benefit clearly outweighs the added subtlety.


# 10. Stacking Multiple Decorators

## 10.1 Recap: Syntax

In [ ]:
@decorator1
@decorator2
def function():
    ...

## 10.2 How Python Interprets This

Exactly equivalent to:

In [ ]:
function = decorator1(decorator2(function))

## 10.3 Which Decorator Runs First (Application Order)

`decorator2` is applied first (it wraps the original, untouched function). `decorator1` is applied second (it wraps whatever `decorator2` produced).

## 10.4 Which Wrapper Executes First (Call Order)

When the fully decorated `function()` is finally called, `decorator1`'s wrapper - the outermost one - runs *first*, since it's the one actually bound to the name `function`. It then calls into `decorator2`'s wrapper, which calls the original function.

## 10.5 Nested Wrappers Visualized

In [ ]:
def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

@bold
@italic
def greet(name):
    return f"Hello, {name}"

print(greet("Ada"))

```
<b><i>Hello, Ada</i></b>
```

`italic` (closest to the function) wraps the raw text first; `bold` (outermost) wraps that already-italicized result. Reversing the order changes the output:

In [ ]:
@italic
@bold
def greet(name):
    return f"Hello, {name}"

print(greet("Ada"))

```
<i><b>Hello, Ada</b></i>
```

## 10.6 Order-Dependent Behavior: A Practical, Consequential Example

In [ ]:
@requires_auth
@log_calls
def delete_account(user):
    ...

Here, the call is logged **before** authentication is checked (since `log_calls` is closer to the function and therefore runs its "before" logic first as execution unwinds inward - wait, tracing carefully: `requires_auth` is outermost, so its wrapper runs first). Let's trace explicitly:

```
requires_auth's wrapper runs -> checks auth -> calls log_calls's wrapper -> logs -> calls delete_account
```

So with this ordering, an unauthenticated call is rejected *before* it is ever logged. Swapping the order:

In [ ]:
@log_calls
@requires_auth
def delete_account(user):
    ...

```
log_calls's wrapper runs -> logs the call -> calls requires_auth's wrapper -> checks auth -> calls delete_account
```

Now, every attempt - authenticated or not - gets logged first. **This is a real, security-relevant decision**, not a stylistic one: which behavior you want (log only successful/authorized calls, or log every attempt including rejected ones) directly determines the correct decorator order.

## 10.7 A General Rule of Thumb

| Goal | Suggested Ordering |
|---|---|
| Security checks should short-circuit before any other work | Put authentication/authorization decorators outermost (topmost `@`). |
| Every attempt, successful or not, should be recorded | Put logging/auditing decorators outermost. |
| Caching should apply to the fully validated, fully processed result | Put caching decorators closer to the function (innermost). |

There is no universally "correct" order - it always depends on which behavior must happen unconditionally versus conditionally. Reasoning through the bottom-up application / top-down execution model (Part 4.7) is the reliable way to predict behavior for any given stack.


# 11. Class-Based Decorators

## 11.1 Why Classes Can Act as Decorators

Anything that is **callable** - meaning it can be used with `()` syntax - can serve as a decorator, since `@decorator` simply requires `decorator(func)` to work. A class instance becomes callable by defining `__call__`, making class-based decorators possible.

## 11.2 \_\_init\_\_ and \_\_call\_\_

In [ ]:
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        print(f"Call #{self.calls} to {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def greet():
    print("Hello")

greet()
greet()
print(greet.calls)   # 2

| Method | Role |
|---|---|
| `__init__(self, func)` | Runs once, at decoration time - receives the original function and stores it. |
| `__call__(self, *args, **kwargs)` | Runs every time the decorated function is *called* - this is where the wrapping behavior lives. |

## 11.3 Callable Objects

An instance of `CountCalls` is not a function - it is an object of type `CountCalls` that happens to *behave* like a function because it implements `__call__`. `greet` above is actually such an instance after decoration, not a plain function object.

## 11.4 Maintaining Decorator State

Class-based decorators are a natural fit when a decorator needs to track state across multiple calls, since `self` provides a clean, conventional place to store it (compare to the function-attribute trick in Part 8.15).

In [ ]:
class RateLimiter:
    def __init__(self, func):
        self.func = func
        self.call_times = []

    def __call__(self, *args, **kwargs):
        import time
        now = time.time()
        self.call_times = [t for t in self.call_times if now - t < 60]
        if len(self.call_times) >= 5:
            raise RuntimeError("Rate limit exceeded")
        self.call_times.append(now)
        return self.func(*args, **kwargs)

@RateLimiter
def call_api():
    ...

## 11.5 Stateful Decorators With Configuration Arguments

In [ ]:
class Retry:
    def __init__(self, max_attempts=3):
        self.max_attempts = max_attempts

    def __call__(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, self.max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception:
                    if attempt == self.max_attempts:
                        raise
            return None
        return wrapper

@Retry(max_attempts=5)
def unreliable():
    ...

Notice this needs a different structure than `CountCalls`: `__init__` here receives the *configuration* (`max_attempts`), and the instance itself - via `__call__` - acts as the "actual decorator," receiving `func` and returning a normal function-based wrapper. This mirrors the three-layer pattern from Part 9, expressed with a class instead of nested functions.

## 11.6 When Class-Based Decorators Are Preferable

| Function-Based Decorators | Class-Based Decorators |
|---|---|
| Simpler for stateless, straightforward wrapping | Clearer when meaningful internal state must be tracked across calls |
| State requires closures or function attributes (less discoverable) | State lives naturally in `self`, following familiar OOP conventions |
| No additional methods needed | Can expose additional methods on the decorator itself (e.g. `.reset()`, `.stats()`) |

In [ ]:
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

    def reset(self):
        self.calls = 0

@CountCalls
def process():
    ...

process()
process.reset()   # a method only possible because the decorator is an object

# 12. Decorators for Classes

## 12.1 Decorators Can Decorate Classes Too

A decorator is not limited to wrapping functions - it can also accept a **class** as its argument and return a (possibly modified) class.

In [ ]:
def decorator(cls):
    # cls is a class object here, not a function
    return cls

## 12.2 The Difference Between Decorating a Function and a Class

| | `@decorator` on a function | `@decorator` on a class |
|---|---|---|
| What is passed in | The function object | The class object itself |
| What is typically returned | A wrapper function | The same class (modified), or an entirely new class |
| Typical use | Wrap behavior around calls | Add attributes/methods, register the class, enforce constraints |

In [ ]:
@decorator
def function():
    ...

In [ ]:
@decorator
class MyClass:
    ...

Both use identical `@` syntax - Python does not distinguish syntactically; it is the decorator's own code that determines what it does with whatever object (function or class) it receives.

## 12.3 Modifying Classes: Adding Attributes

In [ ]:
def add_version(cls):
    cls.version = "1.0"
    return cls

@add_version
class Application:
    pass

print(Application.version)   # "1.0"

## 12.4 Adding Methods to a Class

In [ ]:
def add_greeting(cls):
    def greet(self):
        return f"Hello from {cls.__name__}"
    cls.greet = greet
    return cls

@add_greeting
class Robot:
    pass

r = Robot()
print(r.greet())   # "Hello from Robot"

## 12.5 Registering Classes (Plugin Registration Pattern)

In [ ]:
PLUGIN_REGISTRY = {}

def register_plugin(name):
    def decorator(cls):
        PLUGIN_REGISTRY[name] = cls
        return cls
    return decorator

@register_plugin("csv_exporter")
class CSVExporter:
    def export(self, data):
        ...

@register_plugin("json_exporter")
class JSONExporter:
    def export(self, data):
        ...

print(PLUGIN_REGISTRY)   # {'csv_exporter': <class 'CSVExporter'>, 'json_exporter': <class 'JSONExporter'>}

This is a genuinely common, professional pattern - connecting directly to the plugin architecture discussed in your Modules & Packages handbook, but implemented here declaratively at class-definition time rather than through file discovery.

## 12.6 Validation at Class-Definition Time

In [ ]:
def require_methods(*method_names):
    def decorator(cls):
        for name in method_names:
            if not hasattr(cls, name):
                raise TypeError(f"{cls.__name__} must implement '{name}'")
        return cls
    return decorator

@require_methods("export")
class CSVExporter:
    def export(self, data):
        ...

If `CSVExporter` did not define `export`, this raises a clear, immediate `TypeError` the moment the class is defined - catching a structural mistake early, rather than later when the missing method is finally called.

## 12.7 A Singleton-Style Pattern

In [ ]:
def singleton(cls):
    instances = {}
    @wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class DatabaseConnection:
    def __init__(self):
        print("Creating a new connection")

a = DatabaseConnection()
b = DatabaseConnection()
print(a is b)   # True - the same instance both times

```
Creating a new connection
True
```

**Important nuance:** after this decoration, `DatabaseConnection` is no longer literally a class - it is now `get_instance`, a function. This means `isinstance(a, DatabaseConnection)` would fail, since `DatabaseConnection` is no longer the class itself. Professional code should be aware of this trade-off, and in many cases prefer a metaclass-based or explicit factory-method singleton if `isinstance` checks against the original class are required elsewhere.


# 13. Built-in Python Decorators

## 13.1 Overview

Python and its standard library ship with several decorators that are used constantly in professional code. This section explains what each one does at a practical level; Part 14 goes deeper into the technical mechanics behind `@property`, `@classmethod`, and `@staticmethod`, and Part 17 covers advanced `functools` decorators in full depth.

## 13.2 @staticmethod

In [ ]:
class MathUtils:
    @staticmethod
    def add(a, b):
        return a + b

print(MathUtils.add(2, 3))   # 5 - callable without creating an instance

Marks a method that does not need access to the instance (`self`) or the class (`cls`) at all - it behaves like a plain function that simply happens to live inside a class's namespace for organizational purposes.

## 13.3 @classmethod

In [ ]:
class Pizza:
    def __init__(self, toppings):
        self.toppings = toppings

    @classmethod
    def margherita(cls):
        return cls(["mozzarella", "tomato"])

p = Pizza.margherita()
print(p.toppings)   # ['mozzarella', 'tomato']

Receives the class itself (`cls`) as its first argument instead of an instance (`self`) - commonly used for alternative constructors, as shown here.

## 13.4 @property

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def area(self):
        return 3.14159 * self._radius ** 2

c = Circle(5)
print(c.area)   # 78.53975 - accessed like an attribute, not called like a method

Lets a method be accessed using plain attribute syntax (`c.area`, not `c.area()`), while still running code behind the scenes. Fully explored in Part 14.

## 13.5 @functools.wraps

Already covered in complete depth in Part 7 - preserves a decorated function's identity and metadata.

## 13.6 Other Standard Library Decorators Worth Knowing

| Decorator | Module | Purpose |
|---|---|---|
| `@functools.lru_cache` | `functools` | Automatic memoization (Part 17-18). |
| `@functools.cache` | `functools` | Simplified, unbounded version of `lru_cache` (Python 3.9+). |
| `@functools.total_ordering` | `functools` | Fills in missing comparison methods (`<`, `<=`, `>`, `>=`) from just `__eq__` and one other. |
| `@functools.singledispatch` | `functools` | Enables function overloading based on argument type (Part 17). |
| `@dataclasses.dataclass` | `dataclasses` | Auto-generates `__init__`, `__repr__`, `__eq__` for data-holding classes. |
| `@contextlib.contextmanager` | `contextlib` | Turns a generator function into a context manager usable with `with`. |
| `@abc.abstractmethod` | `abc` | Marks a method that subclasses of an abstract base class must implement. |

In [ ]:
from functools import total_ordering

@total_ordering
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __eq__(self, other):
        return self.amount == other.amount

    def __lt__(self, other):
        return self.amount < other.amount

# total_ordering fills in __le__, __gt__, __ge__ automatically
print(Money(10) > Money(5))   # True

This section intentionally does not re-explain OOP concepts (inheritance, abstract base classes) already covered in your OOP handbook - the focus here is purely on how each decorator itself works.


# 14. property, classmethod, and staticmethod as Decorator Concepts

## 14.1 Why @property Is Useful

`@property` lets a class expose a method as if it were a plain attribute - giving you the syntax simplicity of attribute access, combined with the flexibility of running arbitrary logic (validation, computed values) behind the scenes.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

t = Temperature(0)
print(t.fahrenheit)   # 32.0

## 14.2 Getter, Setter, and Deleter

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):          # getter
        return self._celsius

    @celsius.setter
    def celsius(self, value):   # setter
        if value < -273.15:
            raise ValueError("Below absolute zero")
        self._celsius = value

    @celsius.deleter
    def celsius(self):          # deleter
        print("Deleting celsius value")
        del self._celsius

t = Temperature(25)
t.celsius = 30       # calls the setter
print(t.celsius)     # calls the getter -> 30
del t.celsius         # calls the deleter

| Decorator | Purpose |
|---|---|
| `@property` | Defines the getter - what happens when the attribute is *read*. |
| `@<name>.setter` | Defines what happens when the attribute is *assigned to* (`t.celsius = 30`). |
| `@<name>.deleter` | Defines what happens when `del t.celsius` is used. |

Without a defined setter, attempting `t.celsius = 30` on a property that only has a getter raises `AttributeError: can't set attribute` - a deliberate way to create read-only computed attributes.

## 14.3 @classmethod, Technically

In [ ]:
class Pizza:
    def __init__(self, toppings):
        self.toppings = toppings

    @classmethod
    def margherita(cls):
        return cls(["mozzarella", "tomato"])

`cls` refers to the class itself (here, `Pizza`, or a subclass if called through one) - **not** a specific instance. This is what makes `@classmethod` the correct tool for alternative constructors: `cls(...)` correctly creates an instance of whichever class the method was actually called on, including subclasses.

## 14.4 @staticmethod, Technically

In [ ]:
class MathUtils:
    @staticmethod
    def add(a, b):
        return a + b

A static method receives **neither** `self` nor `cls` automatically - it behaves exactly like a plain, standalone function that happens to be namespaced inside the class for logical grouping.

## 14.5 Differences Between All Three

| | Instance Method | @classmethod | @staticmethod |
|---|---|---|---|
| First parameter | `self` (the instance) | `cls` (the class) | None automatic |
| Can access instance state | Yes | No | No |
| Can access/modify class state | Indirectly, via `self.__class__` | Yes, directly via `cls` | No |
| Typical use | Normal object behavior | Alternative constructors, factory methods | Utility functions logically related to the class |

## 14.6 When to Use Each

| Scenario | Choice |
|---|---|
| The method needs to read or modify instance-specific data | Instance method (no decorator) |
| The method builds and returns a new instance in an alternate way | `@classmethod` |
| The method is a general-purpose utility that doesn't need `self` or `cls` at all, but conceptually belongs with the class | `@staticmethod` |
| The method should be accessed like an attribute, with optional validation on assignment | `@property` |

## 14.7 Common Mistakes

| Mistake | Why It's Wrong |
|---|---|
| Using `@staticmethod` when the method actually needs `cls` or `self` | Leads to hardcoding the class name inside the method body, breaking correctly with subclasses. |
| Forgetting the setter, then trying to assign to a property | Raises `AttributeError: can't set attribute` unexpectedly. |
| Using `@property` for something computationally expensive without documenting it | Callers reasonably expect attribute access to be cheap; an expensive `@property` can surprise them. |
| Defining `@classmethod` but using the concrete class name instead of `cls` inside it | Breaks correct behavior when the method is inherited and called on a subclass. |


# 15. Decorators and Closures

## 15.1 Nested Functions and Enclosing Scopes (Recap)

As introduced in Part 2.6, a function defined inside another function is a nested function. The outer function's local scope is called the **enclosing scope** relative to the inner function.

## 15.2 What a Closure Is

A **closure** occurs when a nested function references a variable from its enclosing scope, and that nested function is returned (or otherwise escapes) beyond the outer function's normal lifetime - yet it still remembers, and can access, that enclosing variable.

In [ ]:
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

c1 = make_counter()
print(c1())   # 1
print(c1())   # 2

c2 = make_counter()
print(c2())   # 1 - a completely independent closure, its own separate "count"

Even though `make_counter()` has already finished executing by the time `c1()` is called, `counter` still has access to `count` - this persistence is exactly what a closure provides.

## 15.3 Free Variables

A **free variable** is a name used inside a nested function that is not a local variable of that nested function, and not a global variable either - it belongs to an enclosing function's scope. In the example above, `count` is a free variable from `counter`'s perspective.

## 15.4 Why Closures Are Fundamental to Decorators

Every decorator wrapper relies on a closure to "remember" the original function:

In [ ]:
def my_decorator(func):        # func is a normal parameter here
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)   # func is a FREE VARIABLE here - captured via closure
    return wrapper

By the time `wrapper` is actually called (potentially long after `my_decorator` itself has finished running), `func` is no longer an "active" parameter in any normal sense - it survives purely because `wrapper` forms a closure over it.

## 15.5 Inspecting a Closure: \_\_closure\_\_ and cell_contents

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def greet():
    print("Hello")

print(greet.__closure__)                        # a tuple of cell objects
print(greet.__closure__[0].cell_contents)        # the original greet function

Each captured free variable is stored in a **cell object**; `cell_contents` gives direct access to the actual value being held onto - useful when debugging decorator behavior at a very low level (Part 25).

## 15.6 Variable Capture and Late Binding: A Classic Trap

In [ ]:
def make_multipliers():
    multipliers = []
    for i in range(3):
        def multiplier(x):
            return x * i     # "i" is looked up when multiplier is CALLED, not when it's defined
        multipliers.append(multiplier)
    return multipliers

funcs = make_multipliers()
print([f(10) for f in funcs])   # [20, 20, 20]  <- NOT [0, 10, 20] as many expect!

Because Python closures capture the **variable itself** (by reference), not its value at the moment the inner function was defined, all three functions end up sharing the *same* `i`, which has finished looping and settled at `2` by the time any of them are actually called.

**The fix - force early binding using a default argument:**

In [ ]:
def make_multipliers():
    multipliers = []
    for i in range(3):
        def multiplier(x, i=i):   # default argument evaluated immediately, at definition time
            return x * i
        multipliers.append(multiplier)
    return multipliers

funcs = make_multipliers()
print([f(10) for f in funcs])   # [0, 10, 20]  correct

## 15.7 Why This Matters Specifically for Decorators

Late binding bugs surface in decorator code most often when a decorator (or decorator factory, Part 9) is applied inside a loop, or when a decorator's inner wrapper captures a loop variable rather than a properly passed-in argument. Being deliberate about what a closure captures - and testing decorators applied in a loop - avoids this entire class of bug.


# 16. Decorators and Scope

## 16.1 The LEGB Rule, Applied to Decorator Code

Python resolves any name by searching, in order: **L**ocal (the current function), **E**nclosing (any outer function), **G**lobal (the module), **B**uilt-in. A decorator's wrapper function typically has all four levels active at once, which is worth tracing deliberately.

In [ ]:
multiplier = 10   # global

def my_decorator(func):
    offset = 5    # enclosing (relative to wrapper)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)   # func: enclosing scope
        return (result * multiplier) + offset   # multiplier: global, offset: enclosing
    return wrapper

| Name | Scope (from wrapper's perspective) |
|---|---|
| `result` | Local to `wrapper` itself |
| `func`, `offset` | Enclosing - captured from `my_decorator`'s scope via closure |
| `multiplier` | Global - looked up in the module's namespace |
| `len`, `print`, etc. | Built-in |

## 16.2 Local Scope Inside a Wrapper

Any variable assigned directly inside `wrapper` (like `result` above) is local to that specific call of `wrapper`, created fresh and destroyed each time, exactly like any ordinary function.

## 16.3 Enclosing Scope and the Decorator's Own Variables

Configuration values passed into a decorator factory (Part 9), like `times` in `repeat(times)`, live in the enclosing scope relative to the wrapper, and are accessed through closure - not passed as an explicit argument to `wrapper` itself.

## 16.4 Global Scope Bugs in Decorators

In [ ]:
call_count = 0

def count_calls(func):
    def wrapper(*args, **kwargs):
        call_count += 1     # UnboundLocalError!
        return func(*args, **kwargs)
    return wrapper

```
UnboundLocalError: local variable 'call_count' referenced before assignment
```

**Why this happens:** because `wrapper` contains an *assignment* to `call_count` (`call_count += 1`), Python treats `call_count` as a local variable for the entire body of `wrapper` - and then the read on the right side of `+=` fails, since that local variable has no value yet at that point.

**The fix - use `global` explicitly, or (better) avoid module-level mutable state:**

In [ ]:
def count_calls(func):
    def wrapper(*args, **kwargs):
        global call_count
        call_count += 1
        return func(*args, **kwargs)
    return wrapper

**A cleaner, more professional fix** - avoid global state entirely by attaching the counter to the function itself (Part 8.15), which sidesteps scope issues altogether.

## 16.5 nonlocal for Enclosing (Non-Global) Mutable State

In [ ]:
def make_counter_decorator():
    count = 0
    def decorator(func):
        def wrapper(*args, **kwargs):
            nonlocal count
            count += 1
            print(f"Call #{count}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

`nonlocal` tells Python that `count` refers to the nearest enclosing (non-global) scope's variable, allowing the wrapper to *modify* it - without `nonlocal`, the same `UnboundLocalError` from Part 16.4 would occur, just one scope level down instead of at global scope.

## 16.6 Summary: Common Scope-Related Decorator Bugs

| Bug | Cause | Fix |
|---|---|---|
| `UnboundLocalError` on a variable the wrapper assigns to | Python treats any assigned-to variable as local, hiding the outer one. | Use `global` or `nonlocal` explicitly, or avoid mutating outer state. |
| Late-binding closure bug (Part 15.6) | A closure captures a variable, not its value at definition time. | Force early binding with a default argument, or restructure to avoid the shared variable. |
| Wrong `func` called after stacking decorators | Confusing which "func" a given wrapper's closure actually refers to after multiple layers of wrapping. | Trace application order carefully (Part 10.2-10.4); use `__wrapped__` (Part 7.5) to inspect. |


# 17. Advanced functools Decorators

## 17.1 functools.wraps (Recap)

Covered fully in Part 7 - preserves metadata on hand-written decorators. Every decorator in this section is itself typically used *alongside* `@wraps` when building custom decorators on top of them.

## 17.2 functools.lru_cache

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(30))   # fast, thanks to caching

`lru_cache` automatically memoizes a function's results, keyed by its arguments, evicting the **L**east **R**ecently **U**sed entry once `maxsize` is exceeded. Fully explored, including caveats, in Part 18.

## 17.3 functools.cache

In [ ]:
from functools import cache

@cache
def expensive_lookup(key):
    ...

Introduced in Python 3.9, `functools.cache` is equivalent to `lru_cache(maxsize=None)` - an unbounded cache with no eviction. Simpler to reach for when memory is not a practical concern for the function's argument space.

## 17.4 functools.cached_property

In [ ]:
from functools import cached_property

class Report:
    def __init__(self, rows):
        self.rows = rows

    @cached_property
    def total(self):
        print("Computing total...")
        return sum(self.rows)

r = Report([1, 2, 3])
print(r.total)   # "Computing total..." then 6
print(r.total)   # 6 - computed only once, cached on the instance

Unlike a plain `@property` (Part 14.1), which recomputes on every access, `cached_property` computes the value once and stores it directly on the instance, replacing itself for that instance going forward. Useful for expensive, immutable-once-computed derived attributes.

**Caveat:** because the cached value is stored as a regular instance attribute after first access, mutating whatever the property depends on (`r.rows` here) will **not** automatically invalidate the cache - a common source of subtle bugs if the underlying data can change.

## 17.5 functools.singledispatch

In [ ]:
from functools import singledispatch

@singledispatch
def describe(value):
    return f"A generic value: {value}"

@describe.register
def _(value: int):
    return f"An integer: {value}"

@describe.register
def _(value: list):
    return f"A list with {len(value)} items"

print(describe(42))        # "An integer: 42"
print(describe([1, 2, 3])) # "A list with 3 items"
print(describe(3.14))      # "A generic value: 3.14"

`singledispatch` implements a form of function overloading based on the type of the *first* argument - dispatching to a different implementation depending on what type is passed, while presenting a single, unified function name to callers.

## 17.6 functools.singledispatchmethod

The same mechanism, adapted for use inside a class on instance methods:

In [ ]:
from functools import singledispatchmethod

class Formatter:
    @singledispatchmethod
    def format(self, value):
        return str(value)

    @format.register
    def _(self, value: int):
        return f"Integer: {value}"

    @format.register
    def _(self, value: str):
        return f"String: '{value}'"

f = Formatter()
print(f.format(42))      # "Integer: 42"
print(f.format("hi"))    # "String: 'hi'"

## 17.7 When Each Should Be Used

| Tool | Use When |
|---|---|
| `lru_cache` | You want automatic caching with a bounded size limit and eviction. |
| `cache` | You want unbounded, simple caching and memory isn't a practical concern. |
| `cached_property` | An instance has an expensive, rarely-changing derived attribute accessed repeatedly. |
| `singledispatch` / `singledispatchmethod` | You need different behavior based on argument type, without a chain of `isinstance` checks. |

## 17.8 Limitations

| Tool | Limitation |
|---|---|
| `lru_cache` / `cache` | All arguments must be hashable; mutable arguments like lists or dicts will raise `TypeError`. |
| `cached_property` | Does not automatically invalidate when underlying instance state changes. |
| `singledispatch` | Dispatches only on the type of the *first* argument, not on any other parameter. |


# 18. Memoization and Caching

## 18.1 What Memoization Is

**Memoization** is the technique of storing the result of an expensive function call, keyed by its arguments, so that future calls with the *same* arguments can return the stored result instantly instead of recomputing it.

## 18.2 Why Caching Improves Performance

Every recomputation of an already-known answer wastes CPU time (or, for I/O-bound calls, wastes time waiting on a network or disk). Caching trades a small amount of memory for a potentially enormous reduction in repeated work - especially valuable for recursive algorithms and expensive external calls.

## 18.3 A Practical Recursive Fibonacci Example

**Without caching - exponential time complexity:**

In [ ]:
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

fibonacci(35)   # noticeably slow - millions of repeated calls

**With @lru_cache - linear time complexity:**

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

fibonacci(35)   # essentially instant

Without caching, `fibonacci(35)` recomputes the same smaller sub-results millions of times. With caching, each unique value of `n` is computed exactly once.

## 18.4 Cache Keys and Hashability Requirements

In [ ]:
@lru_cache
def process(data):
    ...

process([1, 2, 3])

```
TypeError: unhashable type: 'list'
```

`lru_cache` builds its cache key directly from the function's arguments, which requires them to be **hashable** (Part on Data Types/Operators covers hashability in depth). Lists, dictionaries, and sets are not hashable and will raise a `TypeError` immediately.

**The fix - use a hashable equivalent:**

In [ ]:
@lru_cache
def process(data):   # expects a tuple, not a list
    ...

process((1, 2, 3))   # works

## 18.5 Cache Size

In [ ]:
@lru_cache(maxsize=128)   # default
def compute(x):
    ...

@lru_cache(maxsize=None)  # unbounded
def compute_unbounded(x):
    ...

A bounded `maxsize` evicts the least recently used entry once the limit is reached, keeping memory usage predictable. `maxsize=None` (or `@functools.cache`) never evicts anything - appropriate only when the space of distinct arguments is genuinely small or bounded.

## 18.6 Cache Invalidation

`lru_cache` provides no automatic way to invalidate a single stale entry - only the ability to clear the *entire* cache:

In [ ]:
compute.cache_clear()

In [ ]:
print(compute.cache_info())
# CacheInfo(hits=12, misses=4, maxsize=128, currsize=4)

`cache_info()` is invaluable for understanding real cache effectiveness in production - a low hit rate suggests caching may not be providing meaningful benefit for that particular function's usage pattern.

## 18.7 When Caching Is Dangerous

| Danger | Explanation |
|---|---|
| Caching functions with side effects | If a function does more than compute a value (writes to a database, sends a request), caching silently skips those side effects on a cache hit - often not the intended behavior. |
| Caching based on mutable arguments | Even if you convert to a hashable type for the cache key, if the underlying data can change without the object's identity/hash changing, a stale result may be returned. |
| Unbounded caches on high-cardinality inputs | A cache with `maxsize=None` on a function called with effectively infinite distinct arguments (e.g. a unique request ID) grows without bound, causing a memory leak. |
| Caching time-sensitive data | A cached "current price" or "current time" value can become incorrect the moment the real-world value changes, with no automatic expiration. |

## 18.8 Memory Implications

Every cached entry keeps both its argument key and its returned result alive in memory for as long as the cache holds it (or the underlying function object exists, for module-level caches). For large or numerous results, this can meaningfully increase a program's memory footprint - a deliberate `maxsize` and periodic `cache_clear()` (or a proper TTL-aware caching library for time-sensitive data) are professional safeguards.

## 18.9 A Real-World Data/API Example

In [ ]:
from functools import lru_cache
import requests

@lru_cache(maxsize=256)
def get_exchange_rate(currency_code):
    response = requests.get(f"https://api.example.com/rates/{currency_code}")
    response.raise_for_status()
    return response.json()["rate"]

get_exchange_rate("USD")   # hits the real API
get_exchange_rate("USD")   # returns instantly from cache, no network call

This is a genuinely common professional pattern: avoiding redundant, latency-costly, and potentially rate-limited external API calls for data that does not change on every single request within a program's runtime.


# 19. Decorators for Data Science

## 19.1 Measuring Execution Time of Data Pipeline Steps

In [ ]:
import time
from functools import wraps

def timed_step(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[{func.__name__}] {time.perf_counter() - start:.2f}s")
        return result
    return wrapper

@timed_step
def load_dataset(path):
    import pandas as pd
    return pd.read_csv(path)

@timed_step
def clean_dataset(df):
    return df.dropna()

## 19.2 Logging Data-Processing Functions

In [ ]:
def log_shape(func):
    @wraps(func)
    def wrapper(df, *args, **kwargs):
        result = func(df, *args, **kwargs)
        print(f"{func.__name__}: {df.shape} -> {result.shape}")
        return result
    return wrapper

@log_shape
def remove_outliers(df, column, threshold=3):
    z = (df[column] - df[column].mean()) / df[column].std()
    return df[z.abs() < threshold]

## 19.3 Validating Datasets Before Processing

In [ ]:
def validate_columns(*required_columns):
    def decorator(func):
        @wraps(func)
        def wrapper(df, *args, **kwargs):
            missing = [c for c in required_columns if c not in df.columns]
            if missing:
                raise ValueError(f"Missing required columns: {missing}")
            return func(df, *args, **kwargs)
        return wrapper
    return decorator

@validate_columns("customer_id", "purchase_amount")
def compute_revenue(df):
    return df["purchase_amount"].sum()

## 19.4 Tracking Model-Training Time

In [ ]:
@timed_step
def train_model(X_train, y_train):
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    return model

## 19.5 Monitoring Preprocessing Functions (Combining Patterns)

In [ ]:
@timed_step
@log_shape
@validate_columns("age", "income")
def normalize_features(df):
    df = df.copy()
    df["age"] = (df["age"] - df["age"].mean()) / df["age"].std()
    df["income"] = (df["income"] - df["income"].mean()) / df["income"].std()
    return df

Stacking decorators (Part 10) is especially natural in data pipelines: validation runs first (closest to the function), then shape logging, then timing wraps the entire validated, logged operation.

## 19.6 Caching Expensive Computations

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=32)
def compute_correlation_matrix(dataset_id):
    df = load_dataset_by_id(dataset_id)
    return df.corr()

Effective when the same dataset's correlation matrix is requested repeatedly (e.g. across multiple notebook cells or dashboard refreshes) without the underlying data changing.

## 19.7 Checking Function Inputs

In [ ]:
def require_dataframe(func):
    @wraps(func)
    def wrapper(df, *args, **kwargs):
        import pandas as pd
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"{func.__name__} expects a DataFrame, got {type(df)}")
        return func(df, *args, **kwargs)
    return wrapper

## 19.8 Recording Experiment Metadata

In [ ]:
import time
import json

def track_experiment(experiment_name):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.time()
            result = func(*args, **kwargs)
            metadata = {
                "experiment": experiment_name,
                "function": func.__name__,
                "duration": time.time() - start,
                "timestamp": start,
            }
            with open("experiment_log.jsonl", "a") as f:
                f.write(json.dumps(metadata) + "\n")
            return result
        return wrapper
    return decorator

@track_experiment("baseline_model_v1")
def run_experiment(params):
    ...

This pattern gives every experiment run a consistent, automatically recorded audit trail - without cluttering the experiment logic itself with logging code.


# 20. Decorators for Machine Learning and AI Engineering

## 20.1 Model Prediction Logging

In [ ]:
def log_predictions(func):
    @wraps(func)
    def wrapper(model, X, *args, **kwargs):
        predictions = func(model, X, *args, **kwargs)
        print(f"Generated {len(predictions)} predictions")
        return predictions
    return wrapper

@log_predictions
def predict(model, X):
    return model.predict(X)

## 20.2 Model Inference Timing

In [ ]:
@timed_step   # from Part 19.1
def run_inference(model, X):
    return model.predict(X)

Tracking inference latency is essential in production ML systems, where response-time budgets (e.g. under 200ms per request) are often a hard requirement.

## 20.3 Input Validation for Model Serving

In [ ]:
def validate_features(expected_columns):
    def decorator(func):
        @wraps(func)
        def wrapper(model, X, *args, **kwargs):
            missing = set(expected_columns) - set(X.columns)
            if missing:
                raise ValueError(f"Missing features: {missing}")
            return func(model, X, *args, **kwargs)
        return wrapper
    return decorator

@validate_features(["age", "income", "credit_score"])
def predict_risk(model, X):
    return model.predict(X)

## 20.4 Output Validation

In [ ]:
def validate_probabilities(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        probs = func(*args, **kwargs)
        if not all(0.0 <= p <= 1.0 for p in probs):
            raise ValueError("Predicted probabilities out of valid range [0, 1]")
        return probs
    return wrapper

@validate_probabilities
def predict_proba(model, X):
    return model.predict_proba(X)[:, 1]

## 20.5 API Authentication for Model-Serving Endpoints

In [ ]:
def requires_api_key(func):
    @wraps(func)
    def wrapper(request, *args, **kwargs):
        api_key = request.headers.get("X-API-Key")
        if api_key != EXPECTED_API_KEY:
            raise PermissionError("Invalid or missing API key")
        return func(request, *args, **kwargs)
    return wrapper

## 20.6 Retry Mechanisms for Flaky Model Providers

In [ ]:
@retry(max_attempts=3, delay=2)   # from Part 8.8
def call_llm_api(prompt):
    return llm_client.complete(prompt)

Especially important for AI Engineering applications calling external LLM providers, which can experience transient rate limits or timeouts.

## 20.7 Rate Limiting Model Inference Endpoints

In [ ]:
@rate_limit(max_calls=100, period_seconds=60)   # from Part 8.13
def serve_prediction(request):
    ...

## 20.8 Model Monitoring and Drift Detection Hooks

In [ ]:
def monitor_predictions(func):
    @wraps(func)
    def wrapper(model, X, *args, **kwargs):
        predictions = func(model, X, *args, **kwargs)
        record_prediction_distribution(predictions)   # feeds a monitoring dashboard
        return predictions
    return wrapper

## 20.9 Experiment Tracking (ML-Specific)

In [ ]:
def track_ml_experiment(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        log_to_experiment_tracker({
            "function": func.__name__,
            "metrics": result.get("metrics") if isinstance(result, dict) else None,
        })
        return result
    return wrapper

@track_ml_experiment
def train_and_evaluate(X_train, y_train, X_test, y_test):
    model = train_model(X_train, y_train)
    metrics = evaluate(model, X_test, y_test)
    return {"model": model, "metrics": metrics}

## 20.10 Error Handling in AI Pipelines

In [ ]:
def handle_model_errors(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            log_error(f"{func.__name__} failed: {e}")
            return {"error": str(e), "success": False}
    return wrapper

## 20.11 A Complete FastAPI Model-Serving Example

In [ ]:
from fastapi import FastAPI, Request

app = FastAPI()

@app.post("/predict")
@requires_api_key
@validate_features(["age", "income"])
@timed_step
@handle_model_errors
def predict_endpoint(request: Request, X):
    return {"prediction": model.predict(X)}

This single stack demonstrates how decorators compose cleanly in a real AI-serving application: authentication runs outermost (rejecting unauthorized requests immediately), followed by input validation, timing, and error handling wrapping the core prediction logic - each concern isolated in its own reusable decorator (Part 10.7's ordering principles apply directly here).


# 21. Decorators in Web Development

## 21.1 Framework Decorators, Conceptually

Web frameworks use decorators as their primary way of associating a plain Python function with a URL, a permission requirement, or a piece of middleware-like behavior - without requiring you to inherit from a base class or fill in a rigid template structure.

## 21.2 Flask Routing

In [ ]:
from flask import Flask

app = Flask(__name__)

@app.route("/users/<int:user_id>")
def get_user(user_id):
    return {"id": user_id}

`@app.route(...)` is a decorator factory (Part 9): calling `app.route("/users/<int:user_id>")` returns the actual decorator, which then registers `get_user` internally in Flask's URL-to-function mapping and returns the original function largely unmodified, so it can still be called directly (e.g. in tests) if needed.

## 21.3 FastAPI Routing

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"id": user_id}

FastAPI's decorators follow the same conceptual pattern as Flask's, additionally using the function's type hints (`user_id: int`) to perform automatic request validation and API documentation generation - a practical example of decorators and metadata (Part 7) working together.

## 21.4 Authentication and Authorization in Web Frameworks

In [ ]:
from functools import wraps
from flask import request, abort

def login_required(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        token = request.headers.get("Authorization")
        if not is_valid_token(token):
            abort(401)
        return func(*args, **kwargs)
    return wrapper

@app.route("/dashboard")
@login_required
def dashboard():
    return "Welcome!"

This is exactly the authentication pattern from Part 8.3, applied directly to a real route.

## 21.5 Middleware-Like Patterns

While true middleware in most frameworks is a separate mechanism (applied to *every* request), a decorator can achieve similar per-route behavior:

In [ ]:
def with_cors_headers(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        response = func(*args, **kwargs)
        response.headers["Access-Control-Allow-Origin"] = "*"
        return response
    return wrapper

@app.route("/api/data")
@with_cors_headers
def get_data():
    ...

## 21.6 Request Validation

In [ ]:
from functools import wraps

def validate_json_body(*required_fields):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            data = request.get_json() or {}
            missing = [f for f in required_fields if f not in data]
            if missing:
                return {"error": f"Missing fields: {missing}"}, 400
            return func(*args, **kwargs)
        return wrapper
    return decorator

@app.route("/users", methods=["POST"])
@validate_json_body("name", "email")
def create_user():
    ...

## 21.7 API Endpoint Protection: Combining Patterns

In [ ]:
@app.route("/admin/reports")
@login_required
@requires_role("admin")
@rate_limit(max_calls=10, period_seconds=60)
def admin_reports():
    ...

The same decorator-stacking reasoning from Part 10.6 applies here directly: authentication and role checks are placed outermost, so unauthorized or unauthenticated requests are rejected before any rate-limit quota is consumed or any business logic runs.


# 22. Decorators and Asynchronous Python

## 22.1 The Core Difference: Decorating async def

A wrapper built for regular functions does not work correctly around an `async def` function, because calling an async function does not run its body immediately - it returns a **coroutine object** that must be awaited.

In [ ]:
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)   # for an async func, this only creates a coroutine object!
        print(f"{time.perf_counter() - start:.4f}s")
        return result
    return wrapper

@timed
async def fetch_data():
    ...

fetch_data()   # returns a coroutine, and the printed time is meaningless - the body hasn't run yet

## 22.2 Writing an Async-Aware Wrapper

In [ ]:
import time
from functools import wraps

def async_timed(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = await func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

@async_timed
async def fetch_data():
    import asyncio
    await asyncio.sleep(1)
    return "data"

In [ ]:
import asyncio
asyncio.run(fetch_data())

The key difference: `wrapper` itself must also be declared `async def`, and it must `await func(*args, **kwargs)` rather than calling it directly, so that the original coroutine actually runs to completion before the wrapper continues.

## 22.3 Async Logging

In [ ]:
def async_logged(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = await func(*args, **kwargs)
        print(f"{func.__name__} returned {result!r}")
        return result
    return wrapper

## 22.4 Async Retry Logic

In [ ]:
import asyncio

def async_retry(max_attempts=3, delay=1):
    def decorator(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return await func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"Attempt {attempt} failed: {e}")
                    await asyncio.sleep(delay)
        return wrapper
    return decorator

@async_retry(max_attempts=3, delay=2)
async def call_flaky_service():
    ...

## 22.5 A Decorator That Supports Both Sync and Async Functions

Professional libraries sometimes need a single decorator that works correctly on either kind of function:

In [ ]:
import asyncio
import inspect
from functools import wraps

def universal_timed(func):
    if inspect.iscoroutinefunction(func):
        @wraps(func)
        async def async_wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = await func(*args, **kwargs)
            print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
            return result
        return async_wrapper
    else:
        @wraps(func)
        def sync_wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = func(*args, **kwargs)
            print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
            return result
        return sync_wrapper

`inspect.iscoroutinefunction(func)` detects, at decoration time, whether `func` was defined with `async def`, allowing the decorator to build the appropriate kind of wrapper automatically.

## 22.6 Summary: Sync vs Async Decorator Checklist

| Aspect | Synchronous | Asynchronous |
|---|---|---|
| Wrapper declaration | `def wrapper(*args, **kwargs):` | `async def wrapper(*args, **kwargs):` |
| Calling the original function | `func(*args, **kwargs)` | `await func(*args, **kwargs)` |
| Return value | The direct result | The awaited result |
| Common mistake | N/A (baseline case) | Forgetting `await`, which returns an unresolved coroutine object instead of the real result |


# 23. Writing Production-Quality Decorators

## 23.1 Preserve Metadata

Always apply `@wraps(func)` (Part 7). No production decorator should omit this.

## 23.2 Preserve Return Values

Always capture and return `func`'s result (Part 6). A decorator that silently swallows a return value is a serious, hard-to-detect bug.

## 23.3 Preserve Arguments

Always use `*args, **kwargs` (Part 5) unless there is a specific, well-documented reason to restrict the accepted signature.

## 23.4 Avoid Unnecessary Complexity

In [ ]:
# Overengineered for a simple need:
def log(condition=None, level="INFO", formatter=None, prefix="", include_args=True, include_result=True):
    ...

A decorator that offers ten configuration knobs "just in case" is harder to understand, test, and maintain than several small, focused decorators composed together (Part 10). Prefer composability over a single, all-purpose decorator with excessive parameters.

## 23.5 Avoid Hidden Side Effects

A decorator that silently writes files, mutates global state, or makes network calls - without that being obvious from its name or documentation - violates the principle of least surprise. Name decorators for what they visibly do (`@writes_audit_log`, not `@process`).

## 23.6 Make Decorators Composable

A well-designed decorator should stack cleanly with others (Part 10) without unexpected interactions. Avoid decorators that assume they are the *only* decorator applied, or that depend on a specific stacking order to function correctly at all (as opposed to merely producing a different, still-correct result).

## 23.7 Make Decorators Testable

In [ ]:
def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        ...
        return func(*args, **kwargs)
    return wrapper

Because `wrapper` preserves `func`'s call signature and return value transparently, the decorated function can be tested exactly like an undecorated one in most cases - and `func.__wrapped__` (Part 7.5) allows tests to reach the original, undecorated logic directly when needed (Part 26).

## 23.8 Use Meaningful Names

`@retry`, `@requires_auth`, `@cached` communicate intent immediately. `@wrapper1`, `@helper`, `@do_thing` do not - and force every future reader to open the decorator's source just to understand what a decorated function actually does.

## 23.9 Document Decorators

In [ ]:
def retry(max_attempts=3, delay=1):
    """Retry a function on exception, up to max_attempts times.

    Args:
        max_attempts: Maximum number of attempts before giving up.
        delay: Seconds to wait between attempts.

    Raises:
        The last exception encountered, if all attempts fail.
    """
    ...

A decorator's own docstring should explain its *configuration* and *behavior* - since `@wraps` intentionally does not copy this docstring onto the decorated function (the decorated function keeps its own docstring, correctly).

## 23.10 Handle Exceptions Correctly

In [ ]:
def safe(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except SpecificExpectedError as e:
            handle_gracefully(e)
            return None
        # do NOT catch bare `Exception` here unless you genuinely intend
        # to suppress every possible error, including bugs
    return wrapper

Catching overly broad exceptions inside a decorator can silently hide real bugs in the wrapped function - be as specific as possible about which exceptions a decorator is meant to handle.

## 23.11 Avoid Excessive Decorator Stacking

In [ ]:
@a
@b
@c
@d
@e
@f
def do_something():
    ...

Six stacked decorators is a signal to step back: are all six genuinely independent, necessary concerns? Could some be consolidated, or does the function itself need restructuring? Excessive stacking makes execution order (Part 10) harder to reason about and debug.

## 23.12 Understand Performance Overhead

Every decorator adds at least one extra function call per invocation. For functions called millions of times in tight loops, this overhead can become measurable (Part 27) - profile before assuming a decorator's cost is negligible in performance-critical code paths.

## 23.13 When NOT to Use a Decorator

| Situation | Better Alternative |
|---|---|
| The behavior is needed in exactly one place, once | Just write it inline - a decorator adds indirection with no reuse benefit. |
| The "shared behavior" actually differs meaningfully between call sites | A shared decorator forcing artificial uniformity is worse than a few honest, slightly different implementations. |
| The added behavior fundamentally changes the function's contract (e.g. changes what type it returns) | This surprises callers; consider a differently-named wrapper function instead of a transparent-looking decorator. |
| Extremely performance-critical inner-loop code | The added call overhead may not be acceptable; inline the logic instead. |


# 24. Common Decorator Mistakes

## 24.1 Forgetting to Return the Wrapper

**Wrong:**

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    # missing: return wrapper

@my_decorator
def greet():
    print("Hello")

greet()   # TypeError: 'NoneType' object is not callable

**Why it's wrong:** without `return wrapper`, the decorator implicitly returns `None`; `greet` becomes bound to `None` instead of a callable.

**Correct:**

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

**Explanation:** a decorator's job is to *return* something that will replace the original function - forgetting the final `return` is one of the most common first-time mistakes.

## 24.2 Forgetting to Return the Function's Result

Covered fully with examples in Part 6. **Wrong:** `func(*args, **kwargs)` on its own line, discarding the result. **Correct:** `return func(*args, **kwargs)`.

## 24.3 Forgetting `*args`, `**kwargs`

**Wrong:**

In [ ]:
def my_decorator(func):
    def wrapper():
        return func()
    return wrapper

@my_decorator
def add(a, b):
    return a + b

add(2, 3)   # TypeError: wrapper() takes 0 positional arguments but 2 were given

**Correct:** `def wrapper(*args, **kwargs): return func(*args, **kwargs)` (full explanation in Part 5).

## 24.4 Forgetting @wraps

**Wrong:**

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

**Why it's wrong:** `decorated_func.__name__` becomes `"wrapper"`, breaking introspection, documentation tools, and debugging (Part 7).

**Correct:**

In [ ]:
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

## 24.5 Incorrect Decorator Argument Structure

**Wrong:**

In [ ]:
def repeat(func, times):   # func should NOT be here directly
    def wrapper(*args, **kwargs):
        for _ in range(times):
            result = func(*args, **kwargs)
        return result
    return wrapper

@repeat(3)   # fails - repeat expects both func AND times immediately
def greet():
    ...

**Why it's wrong:** `@repeat(3)` calls `repeat(3)` first, but `repeat` as written requires *two* arguments (`func` and `times`) - it's missing the three-layer structure (Part 9).

**Correct:**

In [ ]:
def repeat(times):
    def actual_decorator(func):
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return actual_decorator

## 24.6 Incorrect Decorator Order

Covered fully with a security-relevant example in Part 10.6 - swapping `@requires_auth` and `@log_calls` changes whether unauthenticated attempts get logged.

## 24.7 Accidentally Executing a Function During Decoration

**Wrong:**

In [ ]:
@my_decorator(some_function())   # note the parentheses - CALLS some_function immediately!
def target():
    ...

**Why it's wrong:** `some_function()` runs immediately, at decoration time, and its *return value* (not the function itself) is passed to `my_decorator` - almost never the intended behavior.

**Correct:**

In [ ]:
@my_decorator(some_function)   # pass the function object, not its result
def target():
    ...

## 24.8 Closure Variable Mistakes (Late Binding)

Covered fully with the classic loop example in Part 15.6. **Wrong:** capturing a loop variable directly inside a decorator built in a loop. **Correct:** force early binding with a default argument.

## 24.9 Excessive Nesting

**Wrong:**

In [ ]:
def decorator(func):
    def wrapper(*args, **kwargs):
        def inner_helper():
            def even_more_nested():
                ...
            return even_more_nested()
        return inner_helper()
    return wrapper

**Why it's wrong:** unnecessary nesting inside the wrapper itself (beyond the standard decorator/wrapper structure) makes the code harder to read and debug for no functional benefit.

**Correct:** keep the wrapper's internal logic flat; extract genuinely reusable helper logic into a properly named, top-level function instead.

## 24.10 Hiding Important Program Behavior

**Wrong:**

In [ ]:
@silently_retry_and_ignore_errors
def charge_customer_card(amount):
    ...

**Why it's wrong:** a decorator that silently swallows failures on something as consequential as a payment charge can hide real, business-critical failures from both logs and callers.

**Correct:** be deliberate and explicit about what a decorator suppresses versus what it re-raises, and never silently swallow errors on operations with real-world side effects without very clear logging and intentional design.

## 24.11 Creating Decorators That Are Difficult to Debug

**Wrong:** a decorator with no logging, no `@wraps`, and multiple layers of nested closures, applied five levels deep with other decorators, and no way to inspect the original function.

**Correct:** always use `@wraps` (preserves `__wrapped__` for introspection - Part 25), keep each decorator focused on one concern, and add optional debug logging that can be toggled without needing to modify the decorator's source.


# 25. Debugging Decorators

## 25.1 Using print() to Trace Execution

The simplest debugging technique: temporarily add `print()` statements inside a wrapper to confirm it is actually being entered, and with what arguments.

In [ ]:
def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"DEBUG: entering wrapper for {func.__name__}, args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"DEBUG: leaving wrapper for {func.__name__}, result={result!r}")
        return result
    return wrapper

## 25.2 Using the inspect Module

In [ ]:
import inspect

print(inspect.getsource(my_decorator))     # view a decorator's own source code
print(inspect.isfunction(greet))            # confirm whether a name is still a plain function

## 25.3 inspect.signature

In [ ]:
import inspect

def add(a, b, *, c=0):
    return a + b + c

print(inspect.signature(add))   # (a, b, *, c=0)

Without `@wraps`, a decorated function's `inspect.signature()` incorrectly reports `(*args, **kwargs)` - the wrapper's own signature - rather than the original function's real parameters. This is another concrete, practical reason `@wraps` matters (Part 7): tools like IDEs and API documentation generators rely on `inspect.signature()` to show accurate parameter information.

In [ ]:
@my_decorator   # assume it uses @wraps
def add(a, b, *, c=0):
    return a + b + c

print(inspect.signature(add))   # correctly shows (a, b, *, c=0), thanks to @wraps

## 25.4 Checking \_\_name\_\_ and \_\_doc\_\_

In [ ]:
print(greet.__name__)   # should be "greet", not "wrapper"
print(greet.__doc__)    # should match the original docstring

A quick, immediate sanity check for whether `@wraps` was applied correctly.

## 25.5 Using \_\_wrapped\_\_ to Reach the Original Function

In [ ]:
print(greet.__wrapped__)          # the original, undecorated function object
original_result = greet.__wrapped__()   # calls the function WITHOUT any decorator behavior

This is extremely useful when debugging: it lets you directly compare the decorated behavior against the raw, undecorated behavior, isolating whether a bug originates in the decorator or in the original function itself.

## 25.6 Reading Tracebacks Through Decorators

In [ ]:
def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def divide(a, b):
    return a / b

divide(10, 0)

```
Traceback (most recent call last):
  File "...", line X, in <module>
    divide(10, 0)
  File "...", line Y, in wrapper
    return func(*args, **kwargs)
  File "...", line Z, in divide
    return a / b
ZeroDivisionError: division by zero
```

Notice the traceback shows an extra frame for `wrapper` - this is expected and normal for decorated functions; the actual error location (`divide`, line Z) is still clearly identified further down the traceback.

## 25.7 Determining Which Wrapper Is Executing (Stacked Decorators)

When multiple decorators are stacked (Part 10), temporarily adding a distinct debug print with each decorator's own name to the top of its wrapper is the fastest way to confirm the actual execution order matches what you expect:

In [ ]:
def decorator_a(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Entering decorator_a")
        return func(*args, **kwargs)
    return wrapper

def decorator_b(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Entering decorator_b")
        return func(*args, **kwargs)
    return wrapper

@decorator_a
@decorator_b
def target():
    print("target running")

target()

```
Entering decorator_a
Entering decorator_b
target running
```


# 26. Testing Decorators

## 26.1 Why Decorators Need Dedicated Tests

A decorator is reusable, applied code - a bug in it silently affects every function it decorates. Testing the decorator's behavior directly, separate from any one specific function it happens to be applied to, catches problems early and confirms the decorator behaves correctly in isolation.

## 26.2 Unit Testing a Basic Decorator

In [ ]:
# decorators.py
from functools import wraps

def double_result(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper

In [ ]:
# test_decorators.py
from decorators import double_result

def test_double_result():
    @double_result
    def add(a, b):
        return a + b

    assert add(2, 3) == 10   # (2 + 3) * 2

## 26.3 Testing Arguments Are Passed Through Correctly

In [ ]:
def test_arguments_passed_through():
    received = {}

    @double_result
    def record(a, b, c=0):
        received["a"], received["b"], received["c"] = a, b, c
        return a + b + c

    record(1, 2, c=3)
    assert received == {"a": 1, "b": 2, "c": 3}

## 26.4 Testing Return Values

In [ ]:
def test_return_value_preserved():
    @my_decorator
    def get_data():
        return {"status": "ok"}

    assert get_data() == {"status": "ok"}

## 26.5 Testing Exceptions

In [ ]:
import pytest

def test_decorator_propagates_exceptions():
    @my_decorator
    def failing():
        raise ValueError("boom")

    with pytest.raises(ValueError, match="boom"):
        failing()

Confirms the decorator does not accidentally swallow exceptions it isn't specifically designed to handle (Part 23.10).

## 26.6 Testing Metadata Preservation

In [ ]:
def test_wraps_preserves_metadata():
    @my_decorator
    def documented_function():
        """This is the docstring."""
        pass

    assert documented_function.__name__ == "documented_function"
    assert documented_function.__doc__ == "This is the docstring."

## 26.7 Testing Stacked Decorators

In [ ]:
def test_stacked_decorators_apply_in_correct_order():
    calls = []

    def track(name):
        def decorator(func):
            @wraps(func)
            def wrapper(*args, **kwargs):
                calls.append(name)
                return func(*args, **kwargs)
            return wrapper
        return decorator

    @track("outer")
    @track("inner")
    def target():
        pass

    target()
    assert calls == ["outer", "inner"]

## 26.8 Testing Stateful Decorators

In [ ]:
def test_count_calls_tracks_correctly():
    @CountCalls   # class-based decorator from Part 11.2
    def action():
        pass

    action()
    action()
    action()
    assert action.calls == 3

## 26.9 Testing Async Decorators

In [ ]:
import pytest
import asyncio

@pytest.mark.asyncio
async def test_async_decorator_preserves_result():
    @async_timed   # from Part 22.2
    async def fetch():
        return "data"

    result = await fetch()
    assert result == "data"

Testing async decorators typically requires an async-capable test runner such as `pytest-asyncio`, since `await` can only be used inside an async context.

## 26.10 Testing Decorator Factories With Different Configurations

In [ ]:
def test_retry_gives_up_after_max_attempts():
    attempts = {"count": 0}

    @retry(max_attempts=3, delay=0)
    def always_fails():
        attempts["count"] += 1
        raise RuntimeError("fail")

    with pytest.raises(RuntimeError):
        always_fails()

    assert attempts["count"] == 3

This confirms not just that the decorator "works," but that its specific *configuration* (`max_attempts=3`) is honored correctly - an important distinction when testing decorator factories (Part 9).


# 27. Decorator Performance

## 27.1 Runtime Overhead: Where It Comes From

Every decorator adds at least one additional Python function call between the caller and the original function's actual body. For most applications this overhead is completely negligible - but it is worth understanding precisely where it comes from and when it can matter.

## 27.2 Function-Call Overhead

In [ ]:
def bare():
    return 1

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@decorator
def decorated():
    return 1

Calling `decorated()` involves: one call to `wrapper`, which then makes a second call to the original function - versus a single call for `bare()`. Each stacked decorator (Part 10) adds one more such layer.

## 27.3 Measuring Decorator Overhead

In [ ]:
import timeit

def bare():
    return 1

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@decorator
def decorated():
    return 1

bare_time = timeit.timeit(bare, number=1_000_000)
decorated_time = timeit.timeit(decorated, number=1_000_000)

print(f"Bare:      {bare_time:.4f}s")
print(f"Decorated: {decorated_time:.4f}s")

Running this typically shows the decorated version taking modestly longer overall across a million calls - but for a function that does any meaningful work at all (a database query, a computation, an API call), this fixed overhead becomes proportionally insignificant.

## 27.4 When Decorators Can Negatively Affect Performance

| Scenario | Why It Matters |
|---|---|
| Extremely hot inner loops calling a decorated function millions of times, where the function itself does very little work | The fixed per-call overhead of the wrapper(s) can become a meaningful fraction of total runtime. |
| Deep decorator stacks (many layers) | Each layer adds its own call overhead, compounding. |
| A decorator that itself does expensive work on every call (e.g. re-parsing a config file each time) | The decorator's own logic, not just the extra call, becomes the bottleneck. |
| Decorators that perform I/O (logging to disk, network calls) on every single invocation | I/O is orders of magnitude slower than plain function calls, and can dominate runtime if applied indiscriminately. |

## 27.5 Practical Optimization

| Technique | Benefit |
|---|---|
| Cache expensive setup work outside the wrapper, not inside it | Avoids repeating costly setup (e.g. compiling a regex) on every single call. |
| Use `@lru_cache`/`@cache` to eliminate redundant work entirely (Part 18) | Often provides far bigger performance wins than worrying about wrapper call overhead. |
| Avoid I/O-per-call in decorators used in hot paths; batch or buffer instead | Keeps logging/auditing decorators from becoming the bottleneck themselves. |
| Reserve heavier decorators (auditing, detailed logging) for boundary functions (e.g. API entry points), not tight inner loops | Applies the cost where it's actually needed, not everywhere indiscriminately. |

In [ ]:
# Inefficient: recompiles the pattern on every call
def validate(func):
    @wraps(func)
    def wrapper(text, *args, **kwargs):
        import re
        pattern = re.compile(r"^[a-z]+$")   # recompiled every single call!
        if not pattern.match(text):
            raise ValueError("Invalid input")
        return func(text, *args, **kwargs)
    return wrapper

In [ ]:
# Efficient: compiled once, at decoration time
import re

def validate(func):
    pattern = re.compile(r"^[a-z]+$")   # compiled ONCE, when the decorator is applied
    @wraps(func)
    def wrapper(text, *args, **kwargs):
        if not pattern.match(text):
            raise ValueError("Invalid input")
        return func(text, *args, **kwargs)
    return wrapper

## 27.6 The Practical Takeaway

For the overwhelming majority of professional Python code, decorator overhead is not worth worrying about - the clarity, reuse, and maintainability benefits (Part 1) far outweigh a few microseconds per call. Measure first (Part 27.3), and optimize only when profiling has actually identified a decorator as a genuine bottleneck in a performance-critical path.


# 28. Professional Real-World Projects

## 28.1 Project 1 - Function Execution Timer

**Problem statement:** Provide a reusable way to measure and report how long any function takes to execute, for use across an entire codebase.

**Requirements:** Must work on functions with any signature; must not alter the function's return value; should report time with reasonable precision.

In [ ]:
import time
from functools import wraps

def execution_timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIMER] {func.__name__} completed in {elapsed:.4f} seconds")
        return result
    return wrapper

@execution_timer
def compute_sum(n):
    return sum(range(n))

compute_sum(1_000_000)

**Expected output:**
```
[TIMER] compute_sum completed in 0.0123 seconds
```

**How it works internally:** `time.perf_counter()` is called immediately before and after invoking the wrapped function; the difference gives elapsed wall-clock time with high resolution, unaffected by system clock adjustments.

**Possible improvements:** return the elapsed time alongside the result rather than only printing it; log to a file or monitoring system instead of stdout; add a configurable precision or unit (ms vs s).

**Professional use case:** identifying slow functions during profiling and performance investigations across a codebase, without manually adding timing code to each one.

## 28.2 Project 2 - Logging Decorator

**Problem statement:** Automatically log every call to selected functions, including arguments and results, for observability in production.

In [ ]:
import logging
from functools import wraps

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("app")

def logged(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"CALL {func.__name__} args={args} kwargs={kwargs}")
        try:
            result = func(*args, **kwargs)
            logger.info(f"RETURN {func.__name__} -> {result!r}")
            return result
        except Exception as e:
            logger.error(f"ERROR in {func.__name__}: {e}")
            raise
    return wrapper

@logged
def divide(a, b):
    return a / b

divide(10, 2)

**Expected output (log lines):**
```
2026-01-01 10:00:00 INFO CALL divide args=(10, 2) kwargs={}
2026-01-01 10:00:00 INFO RETURN divide -> 5.0
```

**How it works internally:** the wrapper logs before invocation, then either logs the successful result or logs and re-raises any exception, ensuring failures are never silently lost from the logs.

**Possible improvements:** redact sensitive arguments before logging; make the log level configurable per decorated function; support structured (JSON) logging for log aggregation systems.

**Professional use case:** production observability - tracing exactly what happened, with what inputs, across a distributed system's function calls.

## 28.3 Project 3 - Authentication Decorator

**Problem statement:** Restrict access to certain functions to only authenticated users, in a reusable way.

In [ ]:
from functools import wraps

class AuthenticationError(Exception):
    pass

def require_authentication(func):
    @wraps(func)
    def wrapper(current_user, *args, **kwargs):
        if current_user is None or not current_user.get("authenticated"):
            raise AuthenticationError(f"Authentication required to call {func.__name__}")
        return func(current_user, *args, **kwargs)
    return wrapper

@require_authentication
def view_account_balance(current_user):
    return f"{current_user['name']}'s balance: $1,000"

print(view_account_balance({"name": "Ada", "authenticated": True}))

**Expected output:**
```
Ada's balance: $1,000
```

**How it works internally:** the wrapper inspects the first argument (assumed, by convention, to be the current user context) before allowing the original function to run at all - an unauthenticated call never reaches the real logic.

**Possible improvements:** integrate with a real session/token system rather than a plain dictionary; support role-based checks (Part 8.4) alongside plain authentication; raise a framework-appropriate HTTP error instead of a generic exception in a web context.

**Professional use case:** protecting sensitive operations (viewing financial data, admin actions) consistently across an application.

## 28.4 Project 4 - Retry Decorator

**Problem statement:** Automatically retry a function that may fail due to transient issues (network blips, temporary unavailability), instead of failing immediately.

In [ ]:
import time
from functools import wraps

def retry(max_attempts=3, delay=1, exceptions=(Exception,)):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    print(f"Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_exception
        return wrapper
    return decorator

attempt_counter = {"n": 0}

@retry(max_attempts=3, delay=0.5, exceptions=(ConnectionError,))
def unreliable_network_call():
    attempt_counter["n"] += 1
    if attempt_counter["n"] < 3:
        raise ConnectionError("Temporary network issue")
    return "Success"

print(unreliable_network_call())

**Expected output:**
```
Attempt 1/3 failed: Temporary network issue
Attempt 2/3 failed: Temporary network issue
Success
```

**How it works internally:** the decorator catches only the specified `exceptions` types, retries up to `max_attempts` times with a delay between attempts, and re-raises the last exception if every attempt fails - never silently giving up without informing the caller.

**Possible improvements:** exponential backoff instead of a fixed delay; jitter to avoid synchronized retry storms across many clients; an async version (Part 22.4) for async codebases.

**Professional use case:** resilient API clients calling external services that occasionally experience transient failures.

## 28.5 Project 5 - Validation Decorator

**Problem statement:** Ensure a function's arguments meet defined constraints before the function body ever runs.

In [ ]:
from functools import wraps

def validate_types(**expected_types):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            import inspect
            bound = inspect.signature(func).bind(*args, **kwargs)
            bound.apply_defaults()
            for name, expected_type in expected_types.items():
                value = bound.arguments.get(name)
                if not isinstance(value, expected_type):
                    raise TypeError(
                        f"{func.__name__}() argument '{name}' must be {expected_type.__name__}, "
                        f"got {type(value).__name__}"
                    )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@validate_types(name=str, age=int)
def register_user(name, age):
    return f"Registered {name}, age {age}"

print(register_user("Ada", 30))
register_user("Ada", "thirty")

**Expected output:**
```
Registered Ada, age 30
Traceback (most recent call last):
  ...
TypeError: register_user() argument 'age' must be int, got str
```

**How it works internally:** `inspect.signature(func).bind(*args, **kwargs)` maps the actual call arguments onto the function's named parameters (regardless of whether they were passed positionally or by keyword), allowing the decorator to check each named argument's type against the declared expectation.

**Possible improvements:** support more complex validation rules (ranges, regex patterns, custom validator functions) beyond plain type checks; integrate with a library like `pydantic` for schema-based validation in larger applications.

**Professional use case:** guarding public API boundaries (library functions, endpoint handlers) against malformed input before any real logic executes.

## 28.6 Project 6 - Caching System

**Problem statement:** Build a custom caching decorator with visibility into cache performance, beyond what `@lru_cache` exposes by default.

In [ ]:
from functools import wraps
import time

def cache_with_ttl(ttl_seconds=60):
    def decorator(func):
        cache = {}
        @wraps(func)
        def wrapper(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            now = time.time()
            if key in cache:
                value, timestamp = cache[key]
                if now - timestamp < ttl_seconds:
                    wrapper.hits += 1
                    return value
            result = func(*args, **kwargs)
            cache[key] = (result, now)
            wrapper.misses += 1
            return result
        wrapper.hits = 0
        wrapper.misses = 0
        wrapper.cache_info = lambda: {"hits": wrapper.hits, "misses": wrapper.misses, "size": len(cache)}
        return wrapper
    return decorator

@cache_with_ttl(ttl_seconds=5)
def get_weather(city):
    print(f"Fetching weather for {city}...")
    return f"Sunny in {city}"

print(get_weather("Lagos"))
print(get_weather("Lagos"))
print(get_weather.cache_info())

**Expected output:**
```
Fetching weather for Lagos...
Sunny in Lagos
Sunny in Lagos
{'hits': 1, 'misses': 1, 'size': 1}
```

**How it works internally:** each unique combination of arguments is stored alongside the timestamp it was cached; a cache hit is only returned if the entry is still within `ttl_seconds`, after which it is treated as expired and recomputed - unlike `@lru_cache`, this adds time-based expiration and exposes hit/miss statistics directly.

**Possible improvements:** add a maximum cache size with eviction (mirroring `lru_cache`'s behavior); make the cache thread-safe with a lock for concurrent use; support manual invalidation of a specific key.

**Professional use case:** caching data that becomes stale after a known interval (weather data, exchange rates, feature flags) where an unbounded, non-expiring cache would eventually serve incorrect results.

## 28.7 Project 7 - API Request Monitoring System

**Problem statement:** Track and report on API endpoint usage: call counts, average latency, and error rates.

In [ ]:
from functools import wraps
import time

class APIMonitor:
    def __init__(self):
        self.stats = {}

    def monitor(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            name = func.__name__
            self.stats.setdefault(name, {"calls": 0, "errors": 0, "total_time": 0.0})
            start = time.perf_counter()
            try:
                result = func(*args, **kwargs)
                return result
            except Exception:
                self.stats[name]["errors"] += 1
                raise
            finally:
                self.stats[name]["calls"] += 1
                self.stats[name]["total_time"] += time.perf_counter() - start
        return wrapper

    def report(self):
        for name, data in self.stats.items():
            avg_time = data["total_time"] / data["calls"] if data["calls"] else 0
            error_rate = data["errors"] / data["calls"] if data["calls"] else 0
            print(f"{name}: calls={data['calls']}, avg={avg_time:.4f}s, error_rate={error_rate:.1%}")

monitor = APIMonitor()

@monitor.monitor
def get_users():
    time.sleep(0.01)
    return ["Ada", "Alan"]

@monitor.monitor
def get_orders():
    time.sleep(0.02)
    raise ConnectionError("Service unavailable")

get_users()
get_users()
try:
    get_orders()
except ConnectionError:
    pass

monitor.report()

**Expected output:**
```
get_users: calls=2, avg=0.0101s, error_rate=0.0%
get_orders: calls=1, avg=0.0201s, error_rate=100.0%
```

**How it works internally:** this uses a class-based decorator (Part 11) where `APIMonitor` holds shared, centralized state (`self.stats`) across every function decorated with `@monitor.monitor` - the `finally` block guarantees call counts and timing are recorded whether or not the call succeeded, while errors are separately tallied only on failure.

**Possible improvements:** expose metrics in a format compatible with real monitoring systems (Prometheus, StatsD); add percentile latency tracking (p50, p95, p99) rather than only the average; make it thread-safe for concurrent request handling.

**Professional use case:** lightweight, self-contained API observability without needing to integrate a full external monitoring stack immediately.

## 28.8 Project 8 - Machine Learning Model Monitoring Decorator

**Problem statement:** Track prediction volume, latency, and basic output statistics for a deployed ML model, to detect performance regressions or unusual behavior over time.

In [ ]:
from functools import wraps
import time
import statistics

class ModelMonitor:
    def __init__(self):
        self.latencies = []
        self.prediction_count = 0
        self.predictions = []

    def track(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            predictions = func(*args, **kwargs)
            elapsed = time.perf_counter() - start

            self.latencies.append(elapsed)
            self.prediction_count += len(predictions)
            self.predictions.extend(predictions)

            if elapsed > 0.5:
                print(f"WARNING: {func.__name__} latency {elapsed:.3f}s exceeds threshold")

            return predictions
        return wrapper

    def summary(self):
        return {
            "total_predictions": self.prediction_count,
            "avg_latency": statistics.mean(self.latencies) if self.latencies else 0,
            "avg_prediction_value": statistics.mean(self.predictions) if self.predictions else 0,
        }

monitor = ModelMonitor()

@monitor.track
def predict(model, X):
    time.sleep(0.05)
    return model.predict(X).tolist()

# predict(my_trained_model, X_batch)
# print(monitor.summary())

**How it works internally:** every call to the wrapped prediction function records its latency and accumulates the raw predictions themselves, enabling both operational monitoring (is inference getting slower?) and basic drift-style observation (is the average predicted value shifting unexpectedly over time?) - both from a single, non-intrusive decorator.

**Possible improvements:** persist metrics to a real monitoring/observability backend rather than in-memory lists (which grow unbounded); add proper statistical drift-detection rather than a plain average; separate latency alerting thresholds per model or endpoint.

**Professional use case:** the foundation of production ML monitoring - catching latency regressions and gross behavioral shifts in a deployed model's outputs without modifying the model-serving code itself.


# 29. Decorators Interview Preparation

## 29.1 Beginner Questions

**Q1. What is a decorator in Python?**
A function that takes another function (or class) as input and returns a new, typically extended, version of it - without modifying the original's source code.

**Q2. What does `@my_decorator` above a function definition actually do?**
It is shorthand for `my_function = my_decorator(my_function)` - applied immediately after the function is defined.

**Q3. What is the difference between `func` and `func()`?**
`func` refers to the function object itself; `func()` calls it and refers to whatever it returns.

**Q4. Why do decorators typically define an inner function called a "wrapper"?**
The wrapper is the new function that replaces the original - it runs extra behavior and calls the original function from within itself.

**Q5. What happens if a decorator's wrapper forgets to return the original function's result?**
The decorated function will always return `None`, silently discarding whatever the original function actually computed.

**Q6. Can a decorator be applied to a function that takes arguments?**
Yes - the wrapper must accept `*args, **kwargs` and forward them to the original function.

**Q7. What is `functools.wraps` used for?**
It preserves the decorated function's original `__name__`, `__doc__`, and other metadata, which a plain wrapper would otherwise overwrite.

**Q8. Can you apply more than one decorator to the same function?**
Yes - this is called stacking, and decorators are applied bottom-up, executed top-down (Part 10).

**Q9. What is a higher-order function?**
A function that accepts another function as an argument, returns a function, or both.

**Q10. Is a decorator required to change a function's behavior?**
No - a decorator could simply return the original function unchanged, though that would rarely be useful in practice.

## 29.2 Intermediate Questions

**Q11. Write the equivalent, non-`@`-syntax form of `@my_decorator\ndef greet(): ...`.**
`def greet(): ...` followed by `greet = my_decorator(greet)`.

**Q12. Why must a decorator's wrapper accept `*args, **kwargs` rather than a fixed parameter list?**
So the same decorator works correctly on any function, regardless of its specific signature.

**Q13. What is a closure, and why is it essential to how decorators work?**
A closure is a nested function that retains access to variables from its enclosing scope even after that scope has finished executing; a decorator's wrapper relies on a closure to remember the original function (`func`) between calls.

**Q14. How do you write a decorator that itself accepts configuration arguments (e.g. `@repeat(3)`)?**
Using a three-layer structure: an outer function that accepts the configuration and returns the actual decorator, which accepts the function and returns the wrapper (Part 9).

**Q15. What does `functools.lru_cache` do?**
Automatically memoizes a function's results, keyed by its arguments, evicting the least recently used entry once a size limit is reached.

**Q16. Why must arguments to an `@lru_cache`-decorated function be hashable?**
Because the cache is implemented as a dictionary keyed on the arguments, and dictionary keys must be hashable.

**Q17. What is the difference between `@staticmethod` and `@classmethod`?**
`@staticmethod` receives neither `self` nor `cls`; `@classmethod` receives the class itself (`cls`) as its first argument.

**Q18. What does the `@property` decorator allow you to do?**
Access a method using plain attribute syntax (no parentheses), optionally with a paired setter and deleter for controlled assignment and deletion.

**Q19. What is the late-binding closure trap, and how do you avoid it?**
Closures capture variables by reference, not by value at definition time; in a loop, this can cause all created functions to share the same final value. It is avoided by capturing the value via a default argument.

**Q20. Can a decorator be applied to a class instead of a function?**
Yes - it receives the class object and can modify it, add attributes/methods, or return an entirely different object (Part 12).

## 29.3 Advanced Questions

**Q21. Explain, in detail, what `__wrapped__` is and why it exists.**
An attribute set by `functools.wraps` that points back to the original, undecorated function - allowing introspection tools (and developers) to "see through" a decorator to the real underlying function.

**Q22. How would you write a single decorator that correctly handles both synchronous and asynchronous functions?**
Use `inspect.iscoroutinefunction(func)` at decoration time to choose between building an `async def` wrapper (that awaits `func`) or a regular wrapper (Part 22.5).

**Q23. Explain exactly what happens, step by step, when Python encounters `@decorator1 @decorator2 def f(): ...`.**
Python defines `f`, then computes `decorator2(f)`, then computes `decorator1(decorator2(f))`, and finally rebinds the name `f` to that final result.

**Q24. Why can a decorator that does not use `@wraps` break `inspect.signature()`?**
Because `inspect.signature()` reports on the wrapper's own signature (`*args, **kwargs`) unless `@wraps` restores the original function's signature information via `__wrapped__`.

**Q25. What is the difference between a function-based decorator and a class-based decorator using `__call__`?**
A function-based decorator returns a plain function as its wrapper; a class-based decorator returns a callable *instance*, which can hold state naturally in `self` and expose additional methods.

**Q26. How would you build a decorator that only applies its behavior conditionally, based on an environment variable?**
Check the environment variable inside the wrapper (or even at decoration time, if the check should not vary per call) and branch accordingly, still always returning the correct result either way.

**Q27. What's a subtle risk of using `@functools.cached_property` on a mutable object's derived attribute?**
The cached value is computed once and stored on the instance; if the underlying data changes afterward, the cached property does not automatically recompute, potentially returning stale results.

**Q28. Explain why decorator order matters for a stack combining authentication and logging.**
Whichever decorator is outermost runs its logic first; placing authentication outermost rejects unauthorized calls before they are logged, while placing logging outermost logs every attempt, authorized or not (Part 10.6).

**Q29. How does `functools.singledispatch` achieve type-based function overloading in Python?**
It registers multiple implementations for different types under one shared function name, dispatching to the correct implementation based on the runtime type of the first argument.

**Q30. What happens if you use `@lru_cache` on a method (not a plain function) without care?**
Because `self` is included as part of the cache key, the cache holds a reference to every distinct instance the method was called on, which can prevent those instances from being garbage collected, causing a memory leak.

## 29.4 Expert Questions

**Q31. How would you implement a decorator that measures and enforces a maximum execution time, cancelling the function if it runs too long?**
Using threading with a timeout, or `signal.alarm` on Unix systems, to interrupt execution if the wrapped function exceeds the allotted time - noting that true cancellation of arbitrary running Python code is inherently limited.

**Q32. Explain how a metaclass differs from a class decorator for modifying class behavior, and when you'd choose one over the other.**
A class decorator runs once, after the class body has already been fully executed, and simply modifies or replaces the resulting class object; a metaclass controls the *process* of class creation itself, and applies automatically to every subclass without needing to be reapplied - metaclasses are more powerful but also more invasive and harder to reason about.

**Q33. How would you design a decorator so that its inner wrapper's exceptions still show a clean traceback pointing at the original function, not deeply nested wrapper frames?**
Use `functools.wraps` (which helps tooling but not traceback frames directly), and consider re-raising exceptions with `raise ... from None` or restructuring to minimize unnecessary wrapper-induced frames, while accepting that at least one extra frame for the wrapper itself is generally unavoidable.

**Q34. What are the tradeoffs of implementing caching as a decorator versus as an explicit, separate caching layer/service?**
A decorator is simple and local but ties caching lifetime to the process and offers limited invalidation/sharing across processes; an explicit caching layer (e.g. Redis) is more complex to integrate but supports shared, distributed, and more sophisticated invalidation strategies.

**Q35. How would you make a class-based decorator thread-safe when tracking shared state like a call counter?**
Protect the shared state with a `threading.Lock`, acquiring it before reading/modifying the counter inside `__call__`.

## 29.5 Data Science / ML Questions

**Q36. How would you use a decorator to ensure a data-cleaning function never returns a DataFrame with missing values, without modifying the function itself?**
Wrap it with a decorator that checks `df.isnull().sum().sum() == 0` on the returned DataFrame, raising an error (or logging a warning) if any nulls remain (an output-validation pattern, Part 8.11).

**Q37. Why might `@lru_cache` be inappropriate for caching results of a function that takes a pandas DataFrame as an argument?**
DataFrames are unhashable, so `lru_cache` would immediately raise a `TypeError`; caching would need a different keying strategy (e.g. based on a dataset identifier rather than the DataFrame itself).

**Q38. Describe a decorator you'd use to track experiment metadata (hyperparameters, duration, results) automatically during model training.**
A decorator that records the start time, captures the function's arguments (hyperparameters) and return value (metrics), and writes them to an experiment log or tracking system after each training run (Part 19.8).

**Q39. How could a decorator help detect model input drift in a production ML system?**
By recording summary statistics of each batch of input features on every prediction call, and comparing them over time (or against a baseline) to flag meaningful shifts.

**Q40. What's a risk of applying `@timed_step`-style decorators to every single function in a large data pipeline?**
Excessive logging output can obscure genuinely important information, and I/O-heavy logging on every call (Part 27.4) can itself become a measurable performance cost in tight loops.

## 29.6 Python Backend / Software Engineering Questions

**Q41. How do decorators like `@app.route(...)` work conceptually in frameworks like Flask?**
`app.route(path)` is a decorator factory: calling it returns the actual decorator, which registers the decorated function against the given URL path in the framework's internal routing table and returns the function largely unchanged.

**Q42. How would you write a decorator enforcing that an API endpoint can only be called a limited number of times per minute?**
Track call timestamps (per user/key, in production) in a data structure, reject calls once the count within the trailing time window exceeds the configured limit (Part 8.13).

**Q43. What is the risk of stacking too many decorators on a single endpoint function?**
Reduced readability and harder-to-predict execution order/interactions, and cumulative call overhead (Part 27.4) on high-traffic endpoints.

**Q44. How would you unit test that a `@login_required` decorator correctly rejects unauthenticated requests?**
Call the decorated function directly with a mock/fake unauthenticated user object and assert that the expected exception (or HTTP status) is raised, without needing to spin up a real web server.

**Q45. Explain how you'd build a decorator-based plugin registration system for a backend application.**
A decorator that adds the decorated class or function to a shared registry dictionary at decoration/import time, mirroring the pattern from Part 12.5, so the application can discover all registered plugins by iterating that registry.

**Q46. How would a decorator differ from middleware in a typical web framework?**
A decorator applies to one specific function/route explicitly; middleware applies automatically to every request passing through the framework, without needing to be attached to each route individually.

**Q47. What's the risk of a caching decorator applied to a database-writing function?**
Caching a function that has side effects (like a write) means subsequent calls with the same arguments may skip the actual write entirely, which is almost certainly not the intended behavior (Part 18.7).

**Q48. How would you write a decorator that logs a deprecation warning exactly once per process, rather than on every call?**
Track whether the warning has already been issued using a flag stored as an attribute on the wrapper function (or in an enclosing closure variable), and only call `warnings.warn` the first time.

**Q49. Describe how you would debug a production issue where a decorated function seems to silently return `None` unexpectedly.**
Check whether the decorator's wrapper actually returns `func`'s result (Part 6), and use `func.__wrapped__()` (if `@wraps` is present) to directly compare the decorated and undecorated behavior.

**Q50. Why is it good practice for a decorator library used across a large team to include comprehensive unit tests of its own?**
Because a bug in a shared decorator silently propagates to every function that uses it, potentially across the entire codebase - the "blast radius" of a decorator bug is far larger than that of a typical, single-use function bug.


# 30. Practical Exercises

## 30.1 Level 1 - Beginner

**Exercise 1.** Write a decorator `shout` that prints `"START"` before a function runs and `"END"` after it.

Answer:

In [ ]:
def shout(func):
    def wrapper(*args, **kwargs):
        print("START")
        result = func(*args, **kwargs)
        print("END")
        return result
    return wrapper

**Exercise 2.** Apply `shout` to a function `greet()` that prints `"Hi"`. Show the equivalent non-`@` assignment form.

Answer:

In [ ]:
@shout
def greet():
    print("Hi")

Equivalent to: `greet = shout(greet)`.

**Exercise 3.** What is printed by this code, and why?

In [ ]:
def double(func):
    def wrapper():
        return func() * 2
    return wrapper

@double
def get_number():
    return 5

print(get_number())

Answer: `10`. `wrapper` calls `get_number` (now stored via closure as `func`), gets `5`, and returns `5 * 2`.

**Exercise 4.** Fix this broken decorator so it stops raising `TypeError`.

In [ ]:
def my_decorator(func):
    def wrapper():
        return func()
    return wrapper

@my_decorator
def add(a, b):
    return a + b

add(2, 3)

Answer:

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

**Exercise 5.** What does `greet.__name__` print after this code runs, and why?

In [ ]:
def my_decorator(func):
    def wrapper():
        return func()
    return wrapper

@my_decorator
def greet():
    print("Hi")

Answer: `"wrapper"` - because `@wraps` was not used, so the decorated `greet` is literally the `wrapper` function object.

**Exercise 6.** Write a decorator `positive_only` that raises `ValueError` if the single argument passed to the decorated function is negative.

Answer:

In [ ]:
def positive_only(func):
    def wrapper(value):
        if value < 0:
            raise ValueError("Value must be non-negative")
        return func(value)
    return wrapper

**Exercise 7.** Predict the output.

In [ ]:
def add_one(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + 1
    return wrapper

@add_one
@add_one
def get_value():
    return 10

print(get_value())

Answer: `12`. Each `add_one` layer adds 1, applied twice.

**Exercise 8.** Add `functools.wraps` correctly to the decorator from Exercise 6.

Answer:

In [ ]:
from functools import wraps

def positive_only(func):
    @wraps(func)
    def wrapper(value):
        if value < 0:
            raise ValueError("Value must be non-negative")
        return func(value)
    return wrapper

**Exercise 9.** Write a decorator that prints the name of the function being called, using `func.__name__`.

Answer:

In [ ]:
def announce(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

**Exercise 10.** True or false: `@decorator` and `func = decorator(func)` always produce identical results. Explain.

Answer: True - `@decorator` syntax is defined to be exactly equivalent to that assignment form; there is no behavioral difference.

## 30.2 Level 2 - Intermediate

**Exercise 11.** Write a decorator `uppercase_result` that converts a decorated function's string return value to uppercase.

Answer:

In [ ]:
def uppercase_result(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

**Exercise 12.** Write a decorator factory `multiply_result(n)` that multiplies a function's numeric return value by `n`.

Answer:

In [ ]:
def multiply_result(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs) * n
        return wrapper
    return decorator

**Exercise 13.** Predict the output.

In [ ]:
def a(func):
    def wrapper(*args, **kwargs):
        print("a-before")
        result = func(*args, **kwargs)
        print("a-after")
        return result
    return wrapper

def b(func):
    def wrapper(*args, **kwargs):
        print("b-before")
        result = func(*args, **kwargs)
        print("b-after")
        return result
    return wrapper

@a
@b
def f():
    print("f")

f()

Answer:
```
a-before
b-before
f
b-after
a-after
```

**Exercise 14.** Debug: this decorator factory raises `TypeError: repeat() takes 1 positional argument but 2 were given`. Fix it.

In [ ]:
def repeat(times):
    def wrapper(func):
        for _ in range(times):
            func()
    return wrapper

@repeat(3)
def greet():
    print("hi")

Answer: the structure is missing a proper wrapper *function* to return, and calls `func()` at decoration time instead of at call time.

In [ ]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

**Exercise 15.** Write a closure-based counter decorator that raises an exception if a function is called more than `limit` times.

Answer:

In [ ]:
def limit_calls(limit):
    def decorator(func):
        count = 0
        def wrapper(*args, **kwargs):
            nonlocal count
            count += 1
            if count > limit:
                raise RuntimeError(f"{func.__name__} called more than {limit} times")
            return func(*args, **kwargs)
        return wrapper
    return decorator

**Exercise 16.** What is wrong with this code, and what does it actually print?

In [ ]:
def make_greeters():
    greeters = []
    for name in ["Ada", "Alan", "Grace"]:
        def greet():
            return f"Hello, {name}"
        greeters.append(greet)
    return greeters

for g in make_greeters():
    print(g())

Answer: prints `"Hello, Grace"` three times - a late-binding closure bug (Part 15.6); all three functions share the same `name` variable, which ends up as `"Grace"` after the loop finishes.

**Exercise 17.** Fix Exercise 16 using a default argument.

Answer:

In [ ]:
def make_greeters():
    greeters = []
    for name in ["Ada", "Alan", "Grace"]:
        def greet(name=name):
            return f"Hello, {name}"
        greeters.append(greet)
    return greeters

**Exercise 18.** Write a decorator `only_even` that only allows the decorated function to run if its single integer argument is even; otherwise it should return `None` without calling the function.

Answer:

In [ ]:
def only_even(func):
    def wrapper(n):
        if n % 2 != 0:
            return None
        return func(n)
    return wrapper

**Exercise 19.** Given a stacked decorator scenario, explain (in writing, no code) why swapping the order of `@cache` and `@log_calls` changes what gets logged.

Answer: if `@log_calls` is outermost, every call is logged, including ones that would be served from cache; if `@cache` is outermost, calls that hit the cache never reach `@log_calls`'s wrapper at all, so only actual cache misses get logged.

**Exercise 20.** Write a class-based decorator `Trace` that prints every argument passed to the decorated function, using `__init__` and `__call__`.

Answer:

In [ ]:
class Trace:
    def __init__(self, func):
        self.func = func

    def __call__(self, *args, **kwargs):
        print(f"args={args}, kwargs={kwargs}")
        return self.func(*args, **kwargs)

@Trace
def add(a, b):
    return a + b

## 30.3 Level 3 - Advanced

**Exercise 21.** Write a decorator that caches results using a plain dictionary (not `lru_cache`), supporting only functions with a single hashable argument.

Answer:

In [ ]:
def simple_cache(func):
    cache = {}
    def wrapper(arg):
        if arg not in cache:
            cache[arg] = func(arg)
        return cache[arg]
    return wrapper

**Exercise 22.** Extend Exercise 21 to expose `.cache_clear()` on the wrapper.

Answer:

In [ ]:
def simple_cache(func):
    cache = {}
    def wrapper(arg):
        if arg not in cache:
            cache[arg] = func(arg)
        return cache[arg]
    wrapper.cache_clear = cache.clear
    return wrapper

**Exercise 23.** Write a decorator `require_kwargs` that raises `TypeError` if the decorated function is called with any positional arguments at all (only keyword arguments allowed).

Answer:

In [ ]:
def require_kwargs(func):
    def wrapper(*args, **kwargs):
        if args:
            raise TypeError(f"{func.__name__} only accepts keyword arguments")
        return func(**kwargs)
    return wrapper

**Exercise 24.** Predict the output and explain using LEGB (Part 16).

In [ ]:
value = "global"

def outer():
    value = "enclosing"
    def decorator(func):
        def wrapper():
            return func() + " " + value
        return wrapper
    return decorator

@outer()
def get_value():
    return "local-return"

print(get_value())

Answer: `"local-return enclosing"`. Inside `wrapper`, `value` resolves through the enclosing scope (`outer`'s `value = "enclosing"`), not the global `"global"`.

**Exercise 25.** Write a decorator that only executes the wrapped function on weekdays (Monday-Friday), otherwise raising `RuntimeError`.

Answer:

In [ ]:
import datetime

def weekdays_only(func):
    def wrapper(*args, **kwargs):
        if datetime.datetime.now().weekday() >= 5:
            raise RuntimeError(f"{func.__name__} can only run on weekdays")
        return func(*args, **kwargs)
    return wrapper

**Exercise 26.** Write an async decorator that logs how long an async function takes, using `async_timed` conventions from Part 22.2, but rounds the time to 2 decimal places.

Answer:

In [ ]:
import time
from functools import wraps

def async_timed_rounded(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = await func(*args, **kwargs)
        print(f"{func.__name__}: {round(time.perf_counter() - start, 2)}s")
        return result
    return wrapper

**Exercise 27.** Write a decorator factory `validate_type(expected_type)` for single-argument functions that raises `TypeError` with a clear message if the argument isn't the expected type.

Answer:

In [ ]:
def validate_type(expected_type):
    def decorator(func):
        def wrapper(value):
            if not isinstance(value, expected_type):
                raise TypeError(f"Expected {expected_type.__name__}, got {type(value).__name__}")
            return func(value)
        return wrapper
    return decorator

**Exercise 28.** Using `inspect.iscoroutinefunction`, write a decorator that prints `"async"` or `"sync"` depending on the decorated function's type, without changing its behavior otherwise.

Answer:

In [ ]:
import inspect
from functools import wraps

def announce_type(func):
    kind = "async" if inspect.iscoroutinefunction(func) else "sync"
    if inspect.iscoroutinefunction(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            print(kind)
            return await func(*args, **kwargs)
    else:
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(kind)
            return func(*args, **kwargs)
    return wrapper

**Exercise 29.** Write a decorator that registers every decorated function's name into a module-level list called `REGISTERED_FUNCTIONS`, without altering behavior.

Answer:

In [ ]:
REGISTERED_FUNCTIONS = []

def register(func):
    REGISTERED_FUNCTIONS.append(func.__name__)
    return func

**Exercise 30.** Explain (in writing) why Exercise 29's decorator does not need an inner `wrapper` function at all.

Answer: because it does not need to run any code *around* each call - it only needs to run once, at decoration time, to record the name and then return the original function completely unchanged.

## 30.4 Level 4 - Professional

**Exercise 31.** Build a decorator `enforce_contract(pre=None, post=None)` that runs an optional `pre(args, kwargs)` check before the call and an optional `post(result)` check after, raising `AssertionError` if either check fails.

Answer:

In [ ]:
def enforce_contract(pre=None, post=None):
    def decorator(func):
        def wrapper(*args, **kwargs):
            if pre is not None:
                assert pre(args, kwargs), f"{func.__name__}: precondition failed"
            result = func(*args, **kwargs)
            if post is not None:
                assert post(result), f"{func.__name__}: postcondition failed"
            return result
        return wrapper
    return decorator

@enforce_contract(post=lambda r: r >= 0)
def subtract(a, b):
    return a - b

**Exercise 32.** Modify Project 6 (Caching System, Part 28.6) so that expired entries are actively removed from the dictionary on each call, rather than merely being ignored.

Answer (key change inside `wrapper`):

In [ ]:
expired_keys = [k for k, (_, ts) in cache.items() if now - ts >= ttl_seconds]
for k in expired_keys:
    del cache[k]

**Exercise 33.** Write a decorator that measures peak memory usage of a function call using `tracemalloc`.

Answer:

In [ ]:
import tracemalloc
from functools import wraps

def memory_profile(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        tracemalloc.start()
        result = func(*args, **kwargs)
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        print(f"{func.__name__}: peak memory {peak / 1024:.2f} KB")
        return result
    return wrapper

**Exercise 34.** Design (in writing) a decorator ordering strategy for an endpoint needing: rate limiting, authentication, input validation, and logging of every attempt (including rejected ones). Justify the order.

Answer: `@log_calls` outermost (logs every attempt regardless of outcome), then `@rate_limit` (rejects excess traffic before spending effort on auth), then `@require_authentication`, then `@validate_input` innermost (only validates requests that are both authenticated and within rate limits).

**Exercise 35.** Write a decorator that converts any exception raised by the wrapped function into a standardized dictionary `{"success": False, "error": str(e)}`, and wraps successful results as `{"success": True, "data": result}`.

Answer:

In [ ]:
def standardize_response(func):
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            return {"success": True, "data": result}
        except Exception as e:
            return {"success": False, "error": str(e)}
    return wrapper

**Exercise 36.** Write a pytest test confirming that the decorator from Exercise 35 correctly captures an exception.

Answer:

In [ ]:
def test_standardize_response_captures_exception():
    @standardize_response
    def fails():
        raise ValueError("bad input")

    result = fails()
    assert result == {"success": False, "error": "bad input"}

**Exercise 37.** Write a decorator `deprecated_after(version)` that raises `RuntimeError` if called when a global `CURRENT_VERSION` exceeds the given version, and otherwise just issues a `DeprecationWarning`.

Answer:

In [ ]:
import warnings

CURRENT_VERSION = "1.5"

def deprecated_after(version):
    def decorator(func):
        def wrapper(*args, **kwargs):
            if CURRENT_VERSION > version:
                raise RuntimeError(f"{func.__name__} was removed after version {version}")
            warnings.warn(f"{func.__name__} is deprecated after version {version}", DeprecationWarning)
            return func(*args, **kwargs)
        return wrapper
    return decorator

**Exercise 38.** Explain (in writing) the performance tradeoff of adding a `@log_calls` decorator to a function called 10 million times in a tight loop, and propose one mitigation.

Answer: each call incurs both the wrapper's extra function-call overhead and, more significantly, the cost of the logging I/O itself (Part 27.4) - at 10 million calls, this can dominate total runtime. A mitigation: only log a sample of calls (e.g. every 1000th call), or aggregate and log summary statistics periodically instead of logging every single call.

**Exercise 39.** Write a decorator factory `requires_env(var_name)` that raises `EnvironmentError` if the given environment variable is not set, checked once at decoration time rather than on every call.

Answer:

In [ ]:
import os

def requires_env(var_name):
    def decorator(func):
        if var_name not in os.environ:
            raise EnvironmentError(f"Environment variable '{var_name}' is required to use {func.__name__}")
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator

**Exercise 40.** Design and implement a decorator `@feature(name)` that only runs the decorated function if a feature flag (looked up from a dictionary `FEATURE_FLAGS`) is enabled; otherwise it should return `None` and print a message, without raising an exception.

Answer:

In [ ]:
FEATURE_FLAGS = {"new_dashboard": False}

def feature(name):
    def decorator(func):
        def wrapper(*args, **kwargs):
            if not FEATURE_FLAGS.get(name, False):
                print(f"Feature '{name}' is disabled; skipping {func.__name__}")
                return None
            return func(*args, **kwargs)
        return wrapper
    return decorator

@feature("new_dashboard")
def render_new_dashboard():
    print("Rendering new dashboard")

# 31. Decorators Cheat Sheet and Quick Reference

## 31.1 Basic Decorator Syntax

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result
    return wrapper

@my_decorator
def my_function():
    ...

| Piece | Purpose |
|---|---|
| `func` parameter | The original function being decorated |
| `wrapper` | The replacement function, holding new behavior |
| `return wrapper` | Hands back the wrapper to replace the original |
| `@my_decorator` | Shorthand for `my_function = my_decorator(my_function)` |

## 31.2 `*args` and `**kwargs`

| Syntax | Captures |
|---|---|
| `*args` | Positional arguments, as a tuple |
| `**kwargs` | Keyword arguments, as a dictionary |
| `func(*args, **kwargs)` | Forwards everything received straight through to the original function |

## 31.3 @wraps

In [ ]:
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

Preserves: `__name__`, `__doc__`, `__module__`, `__annotations__`; sets `__wrapped__` to the original function.

## 31.4 Decorators With Arguments (Three-Layer Structure)

In [ ]:
def decorator_factory(config):
    def decorator(func):
        def wrapper(*args, **kwargs):
            ...
            return func(*args, **kwargs)
        return wrapper
    return decorator

@decorator_factory(config_value)
def my_function():
    ...

## 31.5 Stacked Decorators

In [ ]:
@a
@b
def f(): ...

| Aspect | Order |
|---|---|
| Application | Bottom-up: `a(b(f))` |
| Execution | Top-down: `a`'s wrapper logic runs first |

## 31.6 Class-Based Decorators

In [ ]:
class MyDecorator:
    def __init__(self, func):
        self.func = func
    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

@MyDecorator
def my_function():
    ...

## 31.7 Class Decorators (Decorating a Class)

In [ ]:
def decorator(cls):
    cls.new_attribute = "value"
    return cls

@decorator
class MyClass:
    ...

## 31.8 Built-in Decorators

| Decorator | Purpose |
|---|---|
| `@staticmethod` | Method needs neither `self` nor `cls` |
| `@classmethod` | Method receives the class (`cls`) as first argument |
| `@property` | Access a method like a plain attribute |
| `@<prop>.setter` / `@<prop>.deleter` | Define assignment / deletion behavior for a property |

## 31.9 functools Decorators

| Decorator | Purpose |
|---|---|
| `@functools.wraps(func)` | Preserve metadata on a custom decorator |
| `@functools.lru_cache(maxsize=128)` | Bounded automatic memoization |
| `@functools.cache` | Unbounded automatic memoization (3.9+) |
| `@functools.cached_property` | Compute once per instance, cache on the instance |
| `@functools.singledispatch` | Type-based function overloading |
| `@functools.total_ordering` | Fill in comparison methods from `__eq__` + one other |

## 31.10 Caching Quick Facts

| Fact | Detail |
|---|---|
| Cache key | Built from the function's arguments |
| Requirement | All arguments must be hashable |
| Inspect | `.cache_info()` shows hits/misses/size |
| Clear | `.cache_clear()` empties the cache |
| Danger | Never cache functions with meaningful side effects |

## 31.11 Common Patterns Quick Index

| Pattern | Part |
|---|---|
| Logging | 8.1 |
| Timing | 8.2 |
| Authentication / Authorization | 8.3 / 8.4 |
| Validation (input/output) | 8.5 / 8.11 |
| Caching | 8.6, 18 |
| Retry | 8.8 |
| Rate limiting | 8.13 |
| Deprecation | 8.14 |

## 31.12 Common Mistakes Quick Index

| Mistake | Fix |
|---|---|
| Missing `return wrapper` | Always return the wrapper from the decorator |
| Missing `return func(...)` | Always return (not just call) the original function's result |
| Missing `*args, **kwargs` | Always accept them unless deliberately restricting the signature |
| Missing `@wraps` | Always apply it to preserve metadata |
| Wrong decorator-with-arguments structure | Use the full three-layer pattern (Part 9) |
| Calling a function instead of passing it (`@dec(func())`) | Pass the function object, not its call result |

## 31.13 Best Practices Checklist

- [ ] Wrapper uses `*args, **kwargs`
- [ ] Wrapper returns `func`'s result
- [ ] `@wraps(func)` applied
- [ ] Meaningful decorator name
- [ ] Documented configuration (for decorator factories)
- [ ] Exceptions handled deliberately, not silently swallowed
- [ ] Tested independently of any one specific decorated function
- [ ] Reasonable, justified position in any decorator stack

## 31.14 One-Page Professional Quick Reference

In [ ]:
# STANDARD DECORATOR TEMPLATE
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper

# DECORATOR WITH ARGUMENTS TEMPLATE
def my_decorator_factory(option):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator

# CLASS-BASED DECORATOR TEMPLATE
class MyDecorator:
    def __init__(self, func):
        self.func = func
        wraps(func)(self)
    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

# ASYNC DECORATOR TEMPLATE
def my_async_decorator(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        return await func(*args, **kwargs)
    return wrapper

| Symbol | Meaning |
|---|---|
| `@decorator` | `f = decorator(f)` |
| `@decorator(arg)` | `f = decorator(arg)(f)` |
| `@d1 @d2` | `f = d1(d2(f))` |
| `func.__wrapped__` | The original, undecorated function (via `@wraps`) |
| `func.__name__` | Should equal the original function's name (via `@wraps`) |


# Conclusion

Decorators are one of the clearest examples of how a small set of foundational ideas - functions as objects, closures, and higher-order functions - combine to produce a tool with enormous practical reach across professional Python. From a single `@timed` wrapper around one script, to the authentication, validation, and rate-limiting stacks protecting a production API, to `@lru_cache` speeding up a machine learning pipeline, the underlying mechanism never changes: a function receiving another function, and returning something new in its place.

Understanding decorators deeply - not just copying `@wraps(func)` from habit, but knowing exactly what problem it solves; not just stacking decorators, but reasoning precisely about application order versus execution order - is what separates code that merely works from code that is genuinely predictable, debuggable, and safe to extend. Returning to the cheat sheet and one-page quick reference in Part 31, while keeping the underlying reasoning from each part in mind, is the most direct path to genuine fluency with Python decorators.

End of Document

Kajola Gbenga | Data Scientist & AI Engineer